# 市場廣度對 0050 未來報酬預測力（v5 Mean vs Zero）

> **重要：本分析為 exploratory analysis。分位門檻使用完整樣本期間估計，含有樣本內資訊，只用於探索市場廣度與未來報酬的關係形狀，不視為可直接交易的樣本外績效。**
>
> **對 +2、+3、+5、+10 日重疊報酬，以 HAC t-value 與 HAC p-value 為主要統計推論；普通 t-test、普通 p-value 與 annualized Sharpe 僅作描述。**

## 研究問題

市場廣度（上市＋上櫃普通股的漲跌家數）是否對 0050 的近日未來報酬具有預測力？

可能機制包含：

- **Momentum**：資金參與面廣，趨勢延續。
- **Mean reversion**：廣度過強或過弱後，價格反向修正。

正式比較基準：

\[
H_0: E(r \mid group) - E(r \mid non\text{-}group) = 0
\]

Signal date 為 \(t\)。市場廣度只使用 \(t\) 日收盤及以前資訊；未來報酬最早自 \(t\) 日收盤後開始。

## 兩套不同的統計問題

本 notebook 同時回答兩個不同問題：

1. **Group vs Non-group**：訊號日是否相對其他日期有較高或較低報酬。虛無假說為 `E(r | group) - E(r | non-group) = 0`。
2. **Group Mean vs Zero**：訊號日之後的平均報酬本身是否為正或負。虛無假說為 `E(r | group) = 0`。

前者用於評估相對預測力；後者用於評估例如市場大跌後是否平均反彈。即使 group mean 為正，也不代表它優於一般市場日；即使 group vs non-group 顯著，也不代表報酬本身為正。

多日重疊報酬以 HAC 為主要推論。Binomial test 未處理報酬序列的時間依賴，因此只作為勝率的補充性檢定，不取代 HAC。所有新檢定均需進行多重比較修正。


### v3 更新

新增 `Excess return vs unconditional` 分位圖：每個 Q1–Q5 的平均未來報酬扣除同一 target 的全樣本平均報酬，並保留 95% CI 與樣本數標示。原始 absolute return 圖仍保留。


## v4 更新

新增 `up_ratio` 與 `down_ratio`，並沿用既有分位數、極端值、HAC、驗證、繪圖與輸出流程；另新增 predictor 的 Pearson 與 Spearman 相關矩陣，以辨識高度重複的市場廣度訊號。


## v5 更新

- 新增 Group Mean vs Zero 的 one-sample HAC 截距檢定與 HAC 95% CI。
- 新增勝率相對 50% 的雙尾 binomial test。
- 對兩套新 p-value 分別執行 Benjamini–Hochberg FDR 與 Bonferroni 修正。
- 新增 `mean_vs_zero_results`、`all_results_v5` 與 Mean Return vs Zero figures。
- 保留原 Group vs Non-group 統計，兩套問題並列且互不覆蓋。


## 1. 套件與 Config

- 延續原 notebook 的 `CACHE_DIR`、`OUTPUT_DIR`、`REFRESH`、`START_DATE`、`END_DATE` 設計。
- FinLab 欄位可能因資料版本或權限不同而異，因此以候選 key 逐一探測；所有成功與失敗都會印出可讀訊息。
- `AUTO_INSTALL = False`：不自動安裝套件。

In [1]:
# 若環境尚未安裝必要套件，可手動取消註解執行一次。
# %pip install finlab pandas numpy scipy statsmodels matplotlib tqdm openpyxl pyarrow

from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple
import json
import hashlib
import math
import warnings
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from scipy import stats
from pathlib import Path

try:
    import statsmodels.api as sm
    from statsmodels.stats.multitest import multipletests
except ImportError as exc:
    raise ImportError(
        "缺少 statsmodels。請先手動執行上方安裝 cell："
        "%pip install statsmodels"
    ) from exc

try:
    from finlab import data
except ImportError as exc:
    raise ImportError(
        "缺少 finlab。請先手動安裝並完成 FinLab 登入設定。"
    ) from exc


# ============================================================
# 0. Config
# ============================================================

PROJECT_DIR = Path.cwd()

CACHE_DIR = PROJECT_DIR / "cache"
OUTPUT_DIR = PROJECT_DIR / "output" / "market_breadth_0050_study"

CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FIGURE_DIR = OUTPUT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

REFRESH = False
START_DATE = "2011-01-01"
END_DATE = None

TARGET_SYMBOL = "0050"

ANNUAL_RF = 0.02
RANDOM_SEED = 42
SCATTER_SAMPLE_N = 3000

PREDICTORS = [
    "breadth_net_ratio",
    "breadth_log_ad_ratio",
    "up_count",
    "down_count",
    "up_ratio",
    "down_ratio",
]

TARGETS = [
    "ret_c0_o1",
    "ret_c0_c1",
    "ret_o1_c1",
    "ret_o1_o2",
    "ret_o1_c2",
    "ret_o1_c3",
    "ret_o1_c5",
    "ret_o1_c10",
]

PLOT_TARGETS = ["ret_c0_c1", "ret_o1_c2", "ret_o1_c5", "ret_o1_c10"]
PLOT_PREDICTORS = PREDICTORS.copy()
SAVE_ALL_PLOTS = False
SHOW_PLOTS = False

# 價格資料 key：依序嘗試。成功 key 會記錄於 metadata。
FINLAB_KEYS = {
    "stock_close": ["price:收盤價"],
    "security_metadata": [
        "company_basic_info",
        "security_categories",
        "stock_basic_info",
    ],
    "adj_open": [
        "etl:adj_open",
        "price:還原開盤價",
        "price:調整開盤價",
    ],
    "adj_close": [
        "etl:adj_close",
        "price:還原收盤價",
        "price:調整收盤價",
    ],
    "open": ["price:開盤價"],
    "close": ["price:收盤價"],
}

# 若自動辨識 metadata 欄位失敗，請在此指定「實際欄位名稱」。
METADATA_COLUMN_OVERRIDES = {
    "symbol": None,
    "name": None,
    "market": None,
    "security_type": None,
}

# 普通股篩選設定。先以明確商品類型欄位判斷，再以名稱排除。
ALLOWED_MARKET_KEYWORDS = [
    "sii",   # 上市
    "otc",   # 上櫃
    "上市",
    "上櫃",
]
COMMON_STOCK_TYPE_KEYWORDS = ["普通股", "Common Stock", "COMMON"]
EXCLUDE_NAME_PATTERNS = [
    r"ETF", r"ETN", r"權證", r"特別股", r"存託憑證", r"TDR",
    r"受益證券", r"指數投資證券", r"債券", r"REIT",
]
EXCLUDE_SYMBOL_PATTERNS = [
    r"^\d{5,}$",       # 多數權證或非四碼普通股
    r"^[A-Z]",         # 海外/特殊代號，保守排除
]

# 驗證抽樣
MANUAL_BREADTH_DATES = 15
MANUAL_RETURN_SAMPLE_N = 3
VALIDATION_TOL = 1e-10

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 100)

print(f"CACHE_DIR : {CACHE_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"研究期間  : {START_DATE} 至 {END_DATE or '資料最新日'}")

C:\Users\hh483\anaconda3\envs\py311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\hh483\anaconda3\envs\py311\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


CACHE_DIR : C:\Users\hh483\Downloads\00 量化交易研究\cache
OUTPUT_DIR: C:\Users\hh483\Downloads\00 量化交易研究\output\market_breadth_0050_study
研究期間  : 2011-01-01 至 資料最新日


## 2. 共用工具、快取與錯誤診斷

保留並強化原 notebook 的：

- `load_or_download()`
- `limit_date()`
- `safe_div()`
- Parquet 快取
- DatetimeIndex 正規化
- 每個重要步驟的 shape、日期範圍、欄位樣本與原始 exception 訊息

In [2]:
# ============================================================
# 1. Utility
# ============================================================

def normalize_datetime_index(obj: pd.DataFrame | pd.Series, name: str) -> pd.DataFrame | pd.Series:
    out = obj.copy()

    if not isinstance(out.index, pd.DatetimeIndex):
        try:
            out.index = pd.to_datetime(out.index)
        except Exception as exc:
            raise ValueError(
                f"[normalize_datetime_index] {name} 無法轉換為 DatetimeIndex。\n"
                f"index sample={list(out.index[:5])}\n"
                f"original exception={repr(exc)}"
            ) from exc

    out = out.loc[~out.index.duplicated(keep="last")].sort_index()
    return out


def object_diagnostics(obj: Any, name: str) -> str:
    if obj is None:
        return f"name={name}, object=None"

    shape = getattr(obj, "shape", None)
    index = getattr(obj, "index", None)
    columns = getattr(obj, "columns", None)

    if index is not None and len(index) > 0:
        index_range = f"{index.min()} ~ {index.max()}"
    else:
        index_range = "empty/no index"

    if columns is not None:
        column_sample = list(columns[:12])
    elif isinstance(obj, pd.Series):
        column_sample = [obj.name]
    else:
        column_sample = "N/A"

    return (
        f"name={name}, type={type(obj).__name__}, shape={shape}, "
        f"index_range={index_range}, columns_sample={column_sample}"
    )


def raise_step_error(step: str, key: str, obj: Any, exc: Exception) -> None:
    raise RuntimeError(
        f"\n[ERROR]\n"
        f"step={step}\n"
        f"data_key={key}\n"
        f"{object_diagnostics(obj, key)}\n"
        f"original_exception={repr(exc)}\n"
    ) from exc


def load_or_download(
    name: str,
    finlab_key: str,
    refresh: bool = False,
    normalize_time_index: bool = True,
) -> pd.DataFrame:
    path = CACHE_DIR / f"{name}.parquet"
    df = None

    try:
        if path.exists() and not refresh:
            print(f"[CACHE] {name}: {path}")
            df = pd.read_parquet(path)
        else:
            print(f"[DOWNLOAD] {finlab_key}")
            df = data.get(finlab_key)

            if isinstance(df, pd.Series):
                df = df.to_frame(name=df.name or name)

            if not isinstance(df, pd.DataFrame):
                raise TypeError(
                    f"FinLab key {finlab_key} 回傳 {type(df).__name__}，預期 DataFrame/Series。"
                )

            if normalize_time_index:
                df = normalize_datetime_index(df, name)
            df.to_parquet(path)
            print(f"[CACHE SAVE] {path}")

        if normalize_time_index:
            df = normalize_datetime_index(df, name)
        print(f"[LOADED] {object_diagnostics(df, name)}")
        return df

    except Exception as exc:
        raise_step_error("load_or_download", finlab_key, df, exc)


def load_first_available(
    logical_name: str,
    candidate_keys: Sequence[str],
    refresh: bool = False,
    required: bool = True,
    normalize_time_index: bool = True,
) -> Tuple[Optional[pd.DataFrame], Optional[str]]:
    errors = []

    for i, key in enumerate(candidate_keys):
        cache_name = f"{logical_name}_{i}_{key.replace(':', '_').replace('/', '_')}"
        try:
            df = load_or_download(
                cache_name,
                key,
                refresh=refresh,
                normalize_time_index=normalize_time_index,
            )
            print(f"[KEY SELECTED] {logical_name} -> {key}")
            return df, key
        except Exception as exc:
            errors.append(f"{key}: {repr(exc)}")
            print(f"[KEY FAILED] {logical_name} -> {key}")

    if required:
        raise ValueError(
            f"找不到可用的 FinLab key：{logical_name}\n"
            f"候選 keys={list(candidate_keys)}\n"
            f"錯誤摘要：\n" + "\n".join(errors)
        )

    return None, None


def limit_date(df: pd.DataFrame | pd.Series) -> pd.DataFrame | pd.Series:
    out = normalize_datetime_index(df, "limit_date_input")

    if START_DATE is not None:
        out = out.loc[out.index >= pd.to_datetime(START_DATE)]

    if END_DATE is not None:
        out = out.loc[out.index <= pd.to_datetime(END_DATE)]

    return out


def safe_div(a, b):
    if isinstance(b, (pd.Series, pd.DataFrame)):
        denominator = b.replace(0, np.nan)
    else:
        denominator = np.nan if b == 0 else b

    out = a / denominator

    if isinstance(out, (pd.Series, pd.DataFrame)):
        out = out.replace([np.inf, -np.inf], np.nan)

    return out


def safe_numeric(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce")


def flatten_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if isinstance(out.columns, pd.MultiIndex):
        out.columns = [
            "_".join(str(x) for x in tup if str(x) not in ("", "None"))
            for tup in out.columns.to_flat_index()
        ]
    else:
        out.columns = [str(c) for c in out.columns]
    return out


def inspect_finlab_object(df: pd.DataFrame, label: str, rows: int = 5) -> None:
    print(f"\n===== INSPECT: {label} =====")
    print(object_diagnostics(df, label))
    print("dtypes sample:")
    print(df.dtypes.head(12))
    print("head:")
    print(df.head(rows))
    print("tail:")
    print(df.tail(rows))


def check_merge_result(df: pd.DataFrame, label: str, required_cols: Sequence[str]) -> None:
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"[{label}] 缺少欄位：{missing_cols}")

    print(f"\n[MERGE CHECK] {label}: rows={len(df)}, cols={len(df.columns)}")
    for col in required_cols:
        print(f"  {col}: missing={df[col].isna().sum()} ({df[col].isna().mean():.2%})")


def pick_column(
    columns: Iterable,
    explicit: Optional[str],
    candidates: Sequence[str],
    purpose: str,
    required: bool = True,
) -> Optional[str]:
    cols = [str(c) for c in columns]

    if explicit is not None:
        if explicit not in cols:
            raise ValueError(
                f"{purpose} 指定欄位 '{explicit}' 不存在。可用欄位：{cols[:30]}"
            )
        return explicit

    normalized = {str(c).strip().lower().replace(" ", ""): str(c) for c in cols}

    for candidate in candidates:
        key = candidate.strip().lower().replace(" ", "")
        if key in normalized:
            return normalized[key]

    for col in cols:
        col_norm = col.strip().lower().replace(" ", "")
        for candidate in candidates:
            cand_norm = candidate.strip().lower().replace(" ", "")
            if cand_norm in col_norm:
                return col

    if required:
        raise ValueError(
            f"無法自動辨識 {purpose} 欄位。\n"
            f"可用 columns={cols[:50]}\n"
            f"請修改 METADATA_COLUMN_OVERRIDES。"
        )

    return None

## 3. FinLab 資料取得與上市／上櫃普通股篩選

篩選原則：

1. 優先使用 FinLab 的商品類型與市場欄位。
2. 僅保留上市／上櫃。
3. 若有「普通股」類型欄位，明確只保留普通股。
4. 再以證券名稱與代號排除 ETF、ETN、權證、特別股、存託憑證及其他非普通股。
5. 若必要欄位無法辨識，會印出型態、columns、sample，並要求修改 Config，不會默默猜測。

In [3]:
# ============================================================
# 2. Data loading and common-stock filtering
# ============================================================

def load_data(refresh: bool = False) -> Dict[str, Any]:
    loaded: Dict[str, Any] = {"selected_keys": {}}

    stock_close, close_key = load_first_available(
        "stock_close",
        FINLAB_KEYS["stock_close"],
        refresh=refresh,
        required=True,
    )
    stock_close = limit_date(stock_close)
    loaded["stock_close"] = stock_close
    loaded["selected_keys"]["stock_close"] = close_key
    inspect_finlab_object(stock_close, f"stock_close ({close_key})")

    metadata, metadata_key = load_first_available(
        "security_metadata",
        FINLAB_KEYS["security_metadata"],
        refresh=refresh,
        required=True,
        normalize_time_index=False,
    )
    loaded["security_metadata_raw"] = metadata
    loaded["selected_keys"]["security_metadata"] = metadata_key
    inspect_finlab_object(metadata, f"security_metadata ({metadata_key})")

    return loaded


def standardize_security_metadata(raw: pd.DataFrame) -> pd.DataFrame:
    df = flatten_columns(raw)

    # company_basic_info 類資料有時以 RangeIndex + 股票代號欄呈現，
    # 不應套用日期裁切。這裡直接檢查欄位。
    symbol_col = pick_column(
        df.columns,
        METADATA_COLUMN_OVERRIDES["symbol"],
        ["stock_id", "股票代號", "證券代號", "symbol", "code"],
        "股票代號",
        required=False,
    )

    if symbol_col is None:
        index_as_text = pd.Index(df.index).astype(str)
        index_looks_like_symbol = index_as_text.str.match(r"^[0-9A-Za-z]{4,10}$").mean() > 0.8
        if index_looks_like_symbol:
            df = df.copy()
            df.insert(0, "__symbol_from_index__", index_as_text)
            symbol_col = "__symbol_from_index__"
            print("[METADATA] 未找到股票代號欄，改用 index；請人工核對 sample。")
        else:
            raise ValueError(
                "無法辨識股票代號欄，且 index 不像證券代號。"
                f" columns={list(df.columns[:50])}, index_sample={list(df.index[:10])}"
            )
    name_col = pick_column(
        df.columns,
        METADATA_COLUMN_OVERRIDES["name"],
        ["公司簡稱", "公司名稱", "股票名稱", "證券名稱", "name"],
        "證券名稱",
        required=True,
    )
    market_col = pick_column(
        df.columns,
        METADATA_COLUMN_OVERRIDES["market"],
        ["市場別", "市場", "上市櫃", "market", "exchange"],
        "市場別",
        required=True,
    )
    type_col = pick_column(
        df.columns,
        METADATA_COLUMN_OVERRIDES["security_type"],
        ["有價證券別", "證券類別", "商品類型", "security_type", "type"],
        "證券類型",
        required=False,
    )

    keep_cols = [symbol_col, name_col, market_col] + ([type_col] if type_col else [])
    out = df.loc[:, keep_cols].copy()

    rename_map = {
        symbol_col: "symbol",
        name_col: "security_name",
        market_col: "market",
    }
    if type_col:
        rename_map[type_col] = "security_type"

    out = out.rename(columns=rename_map)
    out["symbol"] = out["symbol"].astype(str).str.strip()
    out["security_name"] = out["security_name"].astype(str).str.strip()
    out["market"] = out["market"].astype(str).str.strip()

    if "security_type" in out.columns:
        out["security_type"] = out["security_type"].astype(str).str.strip()

    out = out.drop_duplicates(subset=["symbol"], keep="last")

    print("\n[METADATA MAPPING]")
    print({
        "symbol": symbol_col,
        "name": name_col,
        "market": market_col,
        "security_type": type_col,
    })
    print(out.head(10))
    return out


def filter_common_stocks(
    stock_close: pd.DataFrame,
    metadata_raw: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    try:
        metadata = standardize_security_metadata(metadata_raw)

        # 市場別優先採標準化後的精確值比對，避免 sii/otc/rotc 部分匹配。
        allowed_markets = {
            str(x).strip().casefold() for x in ALLOWED_MARKET_KEYWORDS
        }
        normalized_market = metadata["market"].astype(str).str.strip().str.casefold()
        market_mask = normalized_market.isin(allowed_markets)

        if "security_type" in metadata.columns:
            type_regex = "|".join(re.escape(x) for x in COMMON_STOCK_TYPE_KEYWORDS)
            common_type_mask = metadata["security_type"].str.contains(
                type_regex, case=False, na=False, regex=True
            )
        else:
            common_type_mask = pd.Series(True, index=metadata.index)
            warnings.warn(
                "metadata 無法辨識 security_type 欄位；本次將依市場、名稱與代號規則篩選。"
                "請在輸出 metadata 與 sample 中人工核對。",
                UserWarning,
            )

        name_exclusion_regex = "|".join(EXCLUDE_NAME_PATTERNS)
        excluded_by_name = metadata["security_name"].str.contains(
            name_exclusion_regex, case=False, na=False, regex=True
        )

        symbol_exclusion_regex = "|".join(EXCLUDE_SYMBOL_PATTERNS)
        excluded_by_symbol = metadata["symbol"].str.contains(
            symbol_exclusion_regex, case=False, na=False, regex=True
        )

        metadata["is_allowed_market"] = market_mask
        metadata["is_common_type"] = common_type_mask
        metadata["excluded_by_name"] = excluded_by_name
        metadata["excluded_by_symbol"] = excluded_by_symbol
        metadata["is_in_price_columns"] = metadata["symbol"].isin(stock_close.columns.astype(str))

        metadata["include_common_stock"] = (
            metadata["is_allowed_market"]
            & metadata["is_common_type"]
            & ~metadata["excluded_by_name"]
            & ~metadata["excluded_by_symbol"]
            & metadata["is_in_price_columns"]
        )

        print("\n[MARKET FILTER AUDIT]")
        print(
            pd.DataFrame({
                "metadata_count": metadata.groupby("market", dropna=False).size(),
                "allowed_count": metadata.loc[market_mask]
                    .groupby("market", dropna=False).size(),
                "final_included_count": metadata.loc[
                    metadata["include_common_stock"]
                ].groupby("market", dropna=False).size(),
            }).fillna(0).astype(int)
        )

        selected_symbols = metadata.loc[
            metadata["include_common_stock"], "symbol"
        ].tolist()

        if not selected_symbols:
            raise ValueError(
                "普通股篩選後為 0 檔。請檢查 metadata columns/sample、"
                "ALLOWED_MARKET_KEYWORDS、COMMON_STOCK_TYPE_KEYWORDS、"
                "EXCLUDE_NAME_PATTERNS 與 METADATA_COLUMN_OVERRIDES。"
            )

        common_close = stock_close.loc[:, stock_close.columns.astype(str).isin(selected_symbols)].copy()
        common_close.columns = common_close.columns.astype(str)
        common_close = normalize_datetime_index(common_close, "common_stock_close")
        common_close = common_close.apply(pd.to_numeric, errors="coerce")

        selected_metadata = metadata.loc[metadata["include_common_stock"]].copy()
        excluded_metadata = metadata.loc[~metadata["include_common_stock"]].copy()

        print("\n[COMMON STOCK FILTER]")
        print(f"原始價格欄位數     : {stock_close.shape[1]}")
        print(f"metadata 筆數      : {len(metadata)}")
        print(f"篩選後普通股數     : {common_close.shape[1]}")
        print("市場分布:")
        print(selected_metadata["market"].value_counts(dropna=False).head(20))
        if "security_type" in selected_metadata.columns:
            print("商品類型分布:")
            print(selected_metadata["security_type"].value_counts(dropna=False).head(20))
        print("保留 sample:")
        print(selected_metadata[["symbol", "security_name", "market"]].head(20))
        print("排除 sample:")
        print(excluded_metadata[["symbol", "security_name", "market"]].head(20))

        return common_close, selected_metadata, metadata

    except Exception as exc:
        raise_step_error("filter_common_stocks", "security_metadata", metadata_raw, exc)

## 4. 市場廣度計算

- 上漲：`Close[t] > Close[t-1]`
- 下跌：`Close[t] < Close[t-1]`
- 平盤：`Close[t] == Close[t-1]`
- 缺值不視為平盤。
- `breadth_net_ratio` 分母只含上漲＋下跌，不含平盤。

In [4]:
# ============================================================
# 3. Market breadth
# ============================================================

def build_market_breadth(
    close: pd.DataFrame,
    reasonable_universe_total: Optional[int] = None,
) -> pd.DataFrame:
    try:
        close = normalize_datetime_index(close, "common_stock_close")
        prev_close = close.shift(1)
        valid_pair = close.notna() & prev_close.notna()

        up_mask = valid_pair & close.gt(prev_close)
        down_mask = valid_pair & close.lt(prev_close)
        flat_mask = valid_pair & close.eq(prev_close)

        breadth = pd.DataFrame(index=close.index)
        breadth["up_count"] = up_mask.sum(axis=1).astype("int64")
        breadth["down_count"] = down_mask.sum(axis=1).astype("int64")
        breadth["flat_count"] = flat_mask.sum(axis=1).astype("int64")
        breadth["valid_count"] = valid_pair.sum(axis=1).astype("int64")

        if reasonable_universe_total is None:
            reasonable_universe_total = close.shape[1]

        breadth["universe_total"] = int(reasonable_universe_total)
        breadth["coverage_ratio"] = safe_div(
            breadth["valid_count"],
            breadth["universe_total"],
        )

        directional_count = breadth["up_count"] + breadth["down_count"]
        breadth["breadth_net_ratio"] = safe_div(
            breadth["up_count"] - breadth["down_count"],
            directional_count,
        )
        breadth["up_ratio"] = safe_div(
            breadth["up_count"],
            directional_count,
        )
        breadth["down_ratio"] = safe_div(
            breadth["down_count"],
            directional_count,
        )
        breadth["breadth_log_ad_ratio"] = np.log(
            safe_div(
                breadth["up_count"] + 0.5,
                breadth["down_count"] + 0.5,
            )
        )

        breadth = breadth.replace([np.inf, -np.inf], np.nan)

        identity_ok = (
            breadth["up_count"] + breadth["down_count"] + breadth["flat_count"]
            == breadth["valid_count"]
        )
        if not identity_ok.all():
            bad_dates = identity_ok.index[~identity_ok][:10].tolist()
            raise AssertionError(
                f"up + down + flat != valid_count，異常日期 sample={bad_dates}"
            )

        return breadth

    except Exception as exc:
        raise_step_error("build_market_breadth", "common_stock_close", close, exc)


def predictor_quality_table(df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for col in PREDICTORS:
        s = safe_numeric(df[col])
        finite = s.replace([np.inf, -np.inf], np.nan)

        row = {
            "predictor": col,
            "count": int(finite.notna().sum()),
            "nan_count": int(s.isna().sum()),
            "positive_inf_count": int(np.isposinf(s.to_numpy(dtype=float, na_value=np.nan)).sum()),
            "negative_inf_count": int(np.isneginf(s.to_numpy(dtype=float, na_value=np.nan)).sum()),
            "mean": finite.mean(),
            "std": finite.std(),
            "min": finite.min(),
            "q05": finite.quantile(0.05),
            "q20": finite.quantile(0.20),
            "q40": finite.quantile(0.40),
            "median": finite.median(),
            "q60": finite.quantile(0.60),
            "q80": finite.quantile(0.80),
            "q95": finite.quantile(0.95),
            "max": finite.max(),
        }
        rows.append(row)

    out = pd.DataFrame(rows)
    print("\n[PREDICTOR QUALITY]")
    print(out)
    print("\nBreadth head:")
    print(df[["up_count", "down_count", "flat_count", "valid_count", "coverage_ratio"] + PREDICTORS].head())
    print("\nBreadth tail:")
    print(df[["up_count", "down_count", "flat_count", "valid_count", "coverage_ratio"] + PREDICTORS].tail())
    return out

## 5. 0050 OHLC 與未來報酬

優先嘗試還原開盤價／還原收盤價。若無法取得，回退至一般開盤價／收盤價，並把限制寫入 metadata 與 notebook 輸出。

每個公式都以 signal date \(t\) 對齊，不使用負向 shift 誤判方向。

In [5]:
# ============================================================
# 4. 0050 price and forward returns
# ============================================================

TARGET_METADATA = pd.DataFrame([
    {
        "target_name": "ret_c0_o1",
        "start_point": "Close[t]",
        "end_point": "Open[t+1]",
        "approximate_holding_days": 1,
        "overlapping": False,
        "HAC_lag": 0,
        "formula": "Open[t+1] / Close[t] - 1",
    },
    {
        "target_name": "ret_c0_c1",
        "start_point": "Close[t]",
        "end_point": "Close[t+1]",
        "approximate_holding_days": 1,
        "overlapping": False,
        "HAC_lag": 0,
        "formula": "Close[t+1] / Close[t] - 1",
    },
    {
        "target_name": "ret_o1_c1",
        "start_point": "Open[t+1]",
        "end_point": "Close[t+1]",
        "approximate_holding_days": 1,
        "overlapping": False,
        "HAC_lag": 0,
        "formula": "Close[t+1] / Open[t+1] - 1",
    },
    {
        "target_name": "ret_o1_o2",
        "start_point": "Open[t+1]",
        "end_point": "Open[t+2]",
        "approximate_holding_days": 1,
        "overlapping": False,
        "HAC_lag": 0,
        "formula": "Open[t+2] / Open[t+1] - 1",
    },
    {
        "target_name": "ret_o1_c2",
        "start_point": "Open[t+1]",
        "end_point": "Close[t+2]",
        "approximate_holding_days": 2,
        "overlapping": True,
        "HAC_lag": 1,
        "formula": "Close[t+2] / Open[t+1] - 1",
    },
    {
        "target_name": "ret_o1_c3",
        "start_point": "Open[t+1]",
        "end_point": "Close[t+3]",
        "approximate_holding_days": 3,
        "overlapping": True,
        "HAC_lag": 2,
        "formula": "Close[t+3] / Open[t+1] - 1",
    },
    {
        "target_name": "ret_o1_c5",
        "start_point": "Open[t+1]",
        "end_point": "Close[t+5]",
        "approximate_holding_days": 5,
        "overlapping": True,
        "HAC_lag": 4,
        "formula": "Close[t+5] / Open[t+1] - 1",
    },
    {
        "target_name": "ret_o1_c10",
        "start_point": "Open[t+1]",
        "end_point": "Close[t+10]",
        "approximate_holding_days": 10,
        "overlapping": True,
        "HAC_lag": 9,
        "formula": "Close[t+10] / Open[t+1] - 1",
    },
])


def extract_symbol_series(
    df: pd.DataFrame | pd.Series,
    symbol: str,
    label: str,
) -> pd.Series:
    if isinstance(df, pd.Series):
        print(f"[{label}] Series name={df.name}")
        out = df.copy()
    elif isinstance(df, pd.DataFrame):
        print(f"[{label}] shape={df.shape}, columns sample={list(df.columns[:15])}")
        str_map = {str(c): c for c in df.columns}

        if symbol not in str_map:
            raise ValueError(
                f"{label} 找不到標的 {symbol}。\n"
                f"columns sample={list(df.columns[:50])}"
            )
        out = df[str_map[symbol]].copy()
    else:
        raise TypeError(f"{label} 預期 DataFrame/Series，實際為 {type(df).__name__}")

    out = safe_numeric(out)
    out.name = label
    out = normalize_datetime_index(out, label)
    return out


def load_0050_prices(refresh: bool = False) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    info: Dict[str, Any] = {
        "target_symbol": TARGET_SYMBOL,
        "price_adjustment": None,
        "open_key": None,
        "close_key": None,
        "limitation": None,
    }

    adj_open_df, adj_open_key = load_first_available(
        "adj_open",
        FINLAB_KEYS["adj_open"],
        refresh=refresh,
        required=False,
    )
    adj_close_df, adj_close_key = load_first_available(
        "adj_close",
        FINLAB_KEYS["adj_close"],
        refresh=refresh,
        required=False,
    )

    if adj_open_df is not None and adj_close_df is not None:
        open_df, close_df = adj_open_df, adj_close_df
        open_key, close_key = adj_open_key, adj_close_key
        info["price_adjustment"] = "adjusted/restated"
        info["limitation"] = "使用 FinLab 可取得的還原／調整開收盤價。"
    else:
        open_df, open_key = load_first_available(
            "open",
            FINLAB_KEYS["open"],
            refresh=refresh,
            required=True,
        )
        close_df, close_key = load_first_available(
            "close",
            FINLAB_KEYS["close"],
            refresh=refresh,
            required=True,
        )
        info["price_adjustment"] = "unadjusted"
        info["limitation"] = (
            "無法取得候選還原／調整開收盤價 key，已回退至一般開盤價與收盤價。"
            "除權息可能影響跨日報酬，解讀時必須保留此限制。"
        )
        warnings.warn(info["limitation"], UserWarning)

    inspect_finlab_object(open_df, f"0050 open source ({open_key})")
    inspect_finlab_object(close_df, f"0050 close source ({close_key})")

    open_s = extract_symbol_series(open_df, TARGET_SYMBOL, "open_0050")
    close_s = extract_symbol_series(close_df, TARGET_SYMBOL, "close_0050")

    price = pd.concat([open_s, close_s], axis=1, join="outer")
    price = limit_date(price)
    price = normalize_datetime_index(price, "0050_price")

    info["open_key"] = open_key
    info["close_key"] = close_key

    print("\n[0050 PRICE]")
    print(info)
    print(price.head())
    print(price.tail())
    return price, info


def build_forward_returns(price: pd.DataFrame) -> pd.DataFrame:
    required = ["open_0050", "close_0050"]
    missing = [c for c in required if c not in price.columns]
    if missing:
        raise ValueError(f"build_forward_returns 缺少欄位：{missing}")

    out = price.copy()
    o = out["open_0050"]
    c = out["close_0050"]

    out["ret_c0_o1"] = safe_div(o.shift(-1), c) - 1
    out["ret_c0_c1"] = safe_div(c.shift(-1), c) - 1
    out["ret_o1_c1"] = safe_div(c.shift(-1), o.shift(-1)) - 1
    out["ret_o1_o2"] = safe_div(o.shift(-2), o.shift(-1)) - 1
    out["ret_o1_c2"] = safe_div(c.shift(-2), o.shift(-1)) - 1
    out["ret_o1_c3"] = safe_div(c.shift(-3), o.shift(-1)) - 1
    out["ret_o1_c5"] = safe_div(c.shift(-5), o.shift(-1)) - 1
    out["ret_o1_c10"] = safe_div(c.shift(-10), o.shift(-1)) - 1

    out[TARGETS] = out[TARGETS].replace([np.inf, -np.inf], np.nan)
    return out

## 6. Dataset 建立與資料品質檢查

In [6]:
# ============================================================
# 5. Dataset and quality checks
# ============================================================

def build_dataset(
    breadth: pd.DataFrame,
    forward_returns: pd.DataFrame,
) -> pd.DataFrame:
    try:
        breadth = normalize_datetime_index(breadth, "breadth")
        forward_returns = normalize_datetime_index(forward_returns, "forward_returns")

        before_rows = len(breadth)
        df = breadth.join(
            forward_returns[["open_0050", "close_0050"] + TARGETS],
            how="left",
        )
        df = normalize_datetime_index(df, "market_breadth_dataset")

        if len(df) != before_rows:
            raise AssertionError(
                f"join 後列數改變：before={before_rows}, after={len(df)}"
            )

        df["year"] = df.index.year
        df["month"] = df.index.month
        df.index.name = "date"

        check_merge_result(
            df,
            "breadth + 0050",
            PREDICTORS + ["open_0050", "close_0050"] + TARGETS,
        )
        return df

    except Exception as exc:
        raise_step_error("build_dataset", "breadth_and_0050", breadth, exc)


def build_data_quality_table(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for col in df.columns:
        s = df[col]
        numeric = pd.to_numeric(s, errors="coerce")
        values = numeric.to_numpy(dtype=float, na_value=np.nan)

        rows.append({
            "column": col,
            "dtype": str(s.dtype),
            "row_count": len(s),
            "non_null_count": int(s.notna().sum()),
            "missing_count": int(s.isna().sum()),
            "missing_ratio": s.isna().mean(),
            "positive_inf_count": int(np.isposinf(values).sum()),
            "negative_inf_count": int(np.isneginf(values).sum()),
            "min": numeric.min(),
            "max": numeric.max(),
        })

    return pd.DataFrame(rows)


def build_predictor_describe(df: pd.DataFrame) -> pd.DataFrame:
    describe = df[PREDICTORS].describe(
        percentiles=[0.05, 0.20, 0.40, 0.50, 0.60, 0.80, 0.95]
    ).T.reset_index().rename(columns={"index": "predictor"})
    return describe

def build_predictor_correlations(
    df: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """建立 predictor 的 Pearson 與 Spearman 相關矩陣。"""
    predictor_data = (
        df[PREDICTORS]
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
    )

    pearson = predictor_data.corr(method="pearson")
    spearman = predictor_data.corr(method="spearman")
    pearson.index.name = "predictor"
    spearman.index.name = "predictor"
    return pearson, spearman


## 7. 分位數與極端組

本輪使用完整回測區間估計門檻，因此是樣本內 exploratory grouping。

- Q1–Q5：`pd.qcut(..., duplicates="drop")`
- LOW_5：小於等於第 5 百分位
- HIGH_5：大於等於第 95 百分位

In [7]:
# ============================================================
# 6. Groups and cutoffs
# ============================================================

QUINTILE_ORDER = ["Q1", "Q2", "Q3", "Q4", "Q5"]
EXTREME_ORDER = ["LOW_5", "HIGH_5"]


def build_groups(
    df: pd.DataFrame,
    predictors: Sequence[str] = PREDICTORS,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    group_df = pd.DataFrame(index=df.index)
    cutoff_rows = []

    for predictor in predictors:
        s = safe_numeric(df[predictor]).replace([np.inf, -np.inf], np.nan)
        valid = s.dropna()

        if valid.empty:
            raise ValueError(f"{predictor} 沒有有效資料，無法分組。")

        quantiles = {
            "p05": valid.quantile(0.05),
            "p20": valid.quantile(0.20),
            "p40": valid.quantile(0.40),
            "p60": valid.quantile(0.60),
            "p80": valid.quantile(0.80),
            "p95": valid.quantile(0.95),
        }

        qcut_result = pd.Series(pd.NA, index=s.index, dtype="object")
        try:
            raw_q = pd.qcut(
                valid,
                q=5,
                labels=False,
                duplicates="drop",
            )
        except ValueError as exc:
            raise ValueError(
                f"{predictor} 無法建立 qcut。\n"
                f"unique_count={valid.nunique()}, describe={valid.describe().to_dict()}\n"
                f"original_exception={repr(exc)}"
            ) from exc

        n_bins = int(raw_q.max()) + 1 if raw_q.notna().any() else 0
        if n_bins != 5:
            warnings.warn(
                f"{predictor} 因重複切點僅建立 {n_bins} 個分位組；"
                "結果仍保留，但 Q1–Q5 可能不完整。",
                UserWarning,
            )

        label_map = {i: f"Q{i + 1}" for i in range(n_bins)}
        qcut_result.loc[raw_q.index] = raw_q.map(label_map)
        group_df[f"{predictor}__quintile"] = qcut_result

        low_mask = s.le(quantiles["p05"]) & s.notna()
        high_mask = s.ge(quantiles["p95"]) & s.notna()

        group_df[f"{predictor}__LOW_5"] = low_mask
        group_df[f"{predictor}__HIGH_5"] = high_mask

        for cutoff_name, cutoff_value in quantiles.items():
            cutoff_rows.append({
                "predictor": predictor,
                "cutoff_name": cutoff_name,
                "cutoff_value": cutoff_value,
                "valid_n": len(valid),
                "unique_n": valid.nunique(),
                "estimation_scope": "full_sample_exploratory",
            })

    cutoff_df = pd.DataFrame(cutoff_rows)
    return group_df, cutoff_df

## 8. 統計函式

主要推論：

\[
r_t = lpha + eta \cdot group\_dummy_t + \epsilon_t
\]

其中 \(eta = mean(group) - mean(non	ext{-}group)\)。

- 普通檢定：Welch two-sample t-test。
- 重疊報酬：Newey–West / HAC covariance。
- Cohen's d：明確使用 pooled standard deviation。
- Sharpe 僅為描述性指標。

### v5：Mean vs Zero

本節保留原本 Group vs Non-group 的 HAC regression，並新增 Group Mean vs Zero：對每一個 group 的原始 target return 以只有常數項的 OLS 配合 HAC covariance 檢驗截距是否為 0。另以雙尾 binomial test 檢驗勝率是否不同於 50%。兩者回答不同問題。


In [8]:
# ============================================================
# 7. Statistical functions
# ============================================================

def pooled_standard_deviation(group: pd.Series, non_group: pd.Series) -> float:
    g = group.dropna().astype(float)
    ng = non_group.dropna().astype(float)

    n1, n0 = len(g), len(ng)
    if n1 < 2 or n0 < 2:
        return np.nan

    var1 = g.var(ddof=1)
    var0 = ng.var(ddof=1)

    denominator = n1 + n0 - 2
    if denominator <= 0:
        return np.nan

    pooled_var = ((n1 - 1) * var1 + (n0 - 1) * var0) / denominator
    if not np.isfinite(pooled_var) or pooled_var <= 0:
        return np.nan

    return float(np.sqrt(pooled_var))


def calc_cohen_d(group: pd.Series, non_group: pd.Series) -> float:
    pooled_sd = pooled_standard_deviation(group, non_group)
    if not np.isfinite(pooled_sd) or pooled_sd == 0:
        return np.nan
    return float((group.mean() - non_group.mean()) / pooled_sd)


def calc_sharpe(
    returns: pd.Series,
    holding_days: int,
    annual_rf: float = ANNUAL_RF,
) -> Tuple[float, float]:
    y = returns.dropna().astype(float)
    if len(y) < 2:
        return np.nan, np.nan

    std = y.std(ddof=1)
    if not np.isfinite(std) or std == 0:
        return np.nan, np.nan

    h = max(int(holding_days), 1)
    rf_h = (1 + annual_rf) ** (h / 252) - 1
    non_annualized = (y.mean() - rf_h) / std
    annualized = non_annualized * np.sqrt(252 / h)
    return float(non_annualized), float(annualized)


def calc_hac_test(
    target_return: pd.Series,
    group_dummy: pd.Series,
    hac_lag: int,
) -> Dict[str, float]:
    tmp = pd.concat(
        [
            target_return.rename("target_return"),
            group_dummy.rename("group_dummy"),
        ],
        axis=1,
    ).dropna()

    if len(tmp) < 3 or tmp["group_dummy"].nunique() < 2:
        return {
            "HAC_beta": np.nan,
            "HAC_t_value": np.nan,
            "HAC_p_value": np.nan,
            "HAC_n": len(tmp),
        }

    X = sm.add_constant(tmp["group_dummy"].astype(float), has_constant="add")

    try:
        model = sm.OLS(tmp["target_return"].astype(float), X).fit(
            cov_type="HAC",
            cov_kwds={"maxlags": int(hac_lag)},
        )

        return {
            "HAC_beta": float(model.params["group_dummy"]),
            "HAC_t_value": float(model.tvalues["group_dummy"]),
            "HAC_p_value": float(model.pvalues["group_dummy"]),
            "HAC_n": int(model.nobs),
        }

    except Exception as exc:
        print(
            f"[HAC ERROR] lag={hac_lag}, n={len(tmp)}, "
            f"original_exception={repr(exc)}"
        )
        return {
            "HAC_beta": np.nan,
            "HAC_t_value": np.nan,
            "HAC_p_value": np.nan,
            "HAC_n": len(tmp),
        }


def calc_group_statistics(
    df: pd.DataFrame,
    predictor: str,
    target: str,
    group_name: str,
    group_mask: pd.Series,
    target_meta_row: pd.Series,
    group_type: str,
) -> Dict[str, Any]:
    y = safe_numeric(df[target])
    valid_target = y.notna()

    group_mask = group_mask.reindex(df.index).fillna(False).astype(bool)
    group_valid_mask = group_mask & valid_target
    non_group_valid_mask = (~group_mask) & valid_target

    group_returns = y.loc[group_valid_mask]
    non_group_returns = y.loc[non_group_valid_mask]
    unconditional = y.loc[valid_target]

    n_group = len(group_returns)
    n_non_group = len(non_group_returns)

    if n_group >= 2 and n_non_group >= 2:
        ordinary_t, ordinary_p = stats.ttest_ind(
            group_returns,
            non_group_returns,
            equal_var=False,
            nan_policy="omit",
        )
    else:
        ordinary_t, ordinary_p = np.nan, np.nan

    holding_days = int(target_meta_row["approximate_holding_days"])
    non_ann_sharpe, ann_sharpe = calc_sharpe(group_returns, holding_days)

    hac = calc_hac_test(
        target_return=y.loc[valid_target],
        group_dummy=group_mask.loc[valid_target].astype(int),
        hac_lag=int(target_meta_row["HAC_lag"]),
    )

    group_mean = group_returns.mean()
    non_group_mean = non_group_returns.mean()
    unconditional_mean = unconditional.mean()

    group_win_rate = (group_returns > 0).mean() if n_group else np.nan
    unconditional_win_rate = (unconditional > 0).mean() if len(unconditional) else np.nan

    return {
        "predictor": predictor,
        "target": target,
        "group_type": group_type,
        "group": group_name,
        "observation_count": n_group,
        "mean_ret": group_mean,
        "median_ret": group_returns.median(),
        "std_ret": group_returns.std(ddof=1),
        "ret_q1": group_returns.quantile(0.25),
        "ret_q3": group_returns.quantile(0.75),
        "skewness": group_returns.skew(),
        "kurtosis": group_returns.kurt(),
        "win_rate": group_win_rate,
        "non_annualized_sharpe": non_ann_sharpe,
        "annualized_sharpe": ann_sharpe,
        "unconditional_count": len(unconditional),
        "unconditional_mean_ret": unconditional_mean,
        "mean_ret_minus_unconditional": group_mean - unconditional_mean,
        "unconditional_win_rate": unconditional_win_rate,
        "win_rate_minus_unconditional": group_win_rate - unconditional_win_rate,
        "non_group_count": n_non_group,
        "non_group_mean_ret": non_group_mean,
        "mean_diff_group_non_group": group_mean - non_group_mean,
        "ordinary_t_value": ordinary_t,
        "ordinary_p_value": ordinary_p,
        "HAC_beta": hac["HAC_beta"],
        "HAC_t_value": hac["HAC_t_value"],
        "HAC_p_value": hac["HAC_p_value"],
        "Cohen_d": calc_cohen_d(group_returns, non_group_returns),
        "approximate_holding_days": holding_days,
        "overlapping": bool(target_meta_row["overlapping"]),
        "HAC_lag": int(target_meta_row["HAC_lag"]),
    }


def run_full_analysis(
    df: pd.DataFrame,
    groups: pd.DataFrame,
    target_metadata: pd.DataFrame = TARGET_METADATA,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    meta_lookup = target_metadata.set_index("target_name")
    tasks = []

    for predictor in PREDICTORS:
        quintile_col = f"{predictor}__quintile"

        for target in TARGETS:
            for group_name in QUINTILE_ORDER:
                tasks.append((
                    predictor,
                    target,
                    group_name,
                    groups[quintile_col].eq(group_name),
                    "quintile",
                ))

            for group_name in EXTREME_ORDER:
                tasks.append((
                    predictor,
                    target,
                    group_name,
                    groups[f"{predictor}__{group_name}"],
                    "extreme",
                ))

    rows = []
    for predictor, target, group_name, mask, group_type in tqdm(
        tasks,
        desc="Predictor × target × group statistics",
    ):
        rows.append(
            calc_group_statistics(
                df=df,
                predictor=predictor,
                target=target,
                group_name=group_name,
                group_mask=mask,
                target_meta_row=meta_lookup.loc[target],
                group_type=group_type,
            )
        )

    all_results = pd.DataFrame(rows)
    quintile_results = all_results.loc[
        all_results["group_type"].eq("quintile")
    ].reset_index(drop=True)
    extreme_results = all_results.loc[
        all_results["group_type"].eq("extreme")
    ].reset_index(drop=True)

    return quintile_results, extreme_results, all_results

# ============================================================
# v5. Mean vs Zero statistical functions
# ============================================================

def calc_hac_mean_vs_zero(
    group_returns: pd.Series,
    hac_lag: int,
) -> Dict[str, float]:
    """One-sample HAC intercept test: H0 mean(group return) = 0."""
    y = safe_numeric(group_returns).dropna().astype(float)
    n = len(y)
    empty_result = {
        "HAC_mean_vs_zero_alpha": np.nan,
        "HAC_mean_vs_zero_se": np.nan,
        "HAC_mean_vs_zero_t": np.nan,
        "HAC_mean_vs_zero_p": np.nan,
        "mean_ret_ci_lower": np.nan,
        "mean_ret_ci_upper": np.nan,
        "HAC_mean_vs_zero_n": n,
    }
    if n < 2:
        return empty_result

    X = np.ones((n, 1), dtype=float)
    try:
        model = sm.OLS(y.to_numpy(dtype=float), X).fit(
            cov_type="HAC",
            cov_kwds={"maxlags": int(hac_lag)},
        )
        alpha = float(model.params[0])
        se = float(model.bse[0])
        t_value = float(model.tvalues[0])
        p_value = float(model.pvalues[0])
        if not np.isfinite(se) or se < 0:
            return empty_result
        ci_half = 1.96 * se
        return {
            "HAC_mean_vs_zero_alpha": alpha,
            "HAC_mean_vs_zero_se": se,
            "HAC_mean_vs_zero_t": t_value,
            "HAC_mean_vs_zero_p": p_value,
            "mean_ret_ci_lower": alpha - ci_half,
            "mean_ret_ci_upper": alpha + ci_half,
            "HAC_mean_vs_zero_n": int(model.nobs),
        }
    except Exception as exc:
        print(
            f"[HAC MEAN VS ZERO ERROR] lag={hac_lag}, n={n}, "
            f"original_exception={repr(exc)}"
        )
        return empty_result


def calc_binomial_test(group_returns: pd.Series) -> Dict[str, float]:
    """Two-sided test of H0: P(return > 0 | group) = 0.5."""
    y = safe_numeric(group_returns).dropna().astype(float)
    trials = int(len(y))
    successes = int((y > 0).sum())
    failures = int(trials - successes)
    win_rate = successes / trials if trials else np.nan
    p_value = np.nan
    if trials > 0:
        try:
            p_value = float(
                stats.binomtest(
                    k=successes,
                    n=trials,
                    p=0.5,
                    alternative="two-sided",
                ).pvalue
            )
        except Exception as exc:
            print(
                f"[BINOMIAL ERROR] successes={successes}, trials={trials}, "
                f"original_exception={repr(exc)}"
            )
    return {
        "binomial_successes": successes,
        "binomial_failures": failures,
        "binomial_trials": trials,
        "win_rate_minus_50pct": win_rate - 0.5 if trials else np.nan,
        "binomial_p_value": p_value,
    }


def apply_multiple_testing_correction(
    df: pd.DataFrame,
    p_col: str,
    prefix: str,
    alpha: float = 0.05,
) -> Tuple[pd.DataFrame, Dict[str, int]]:
    """Apply BH-FDR and Bonferroni to non-missing p-values only."""
    out = df.copy()
    valid = out[p_col].notna() & np.isfinite(out[p_col].astype(float))
    n_tests = int(valid.sum())

    q_col = f"{prefix}_FDR_q"
    bonf_col = f"{prefix}_Bonferroni_p"
    raw_sig_col = f"{prefix}_significant_raw"
    fdr_sig_col = f"{prefix}_significant_FDR"
    bonf_sig_col = f"{prefix}_significant_Bonferroni"

    out[q_col] = np.nan
    out[bonf_col] = np.nan
    out[raw_sig_col] = False
    out[fdr_sig_col] = False
    out[bonf_sig_col] = False

    if n_tests > 0:
        p_values = out.loc[valid, p_col].astype(float).to_numpy()
        reject_fdr, q_values, _, _ = multipletests(
            p_values,
            alpha=alpha,
            method="fdr_bh",
        )
        bonf_values = np.minimum(p_values * n_tests, 1.0)
        out.loc[valid, q_col] = q_values
        out.loc[valid, bonf_col] = bonf_values
        out.loc[valid, raw_sig_col] = p_values < alpha
        out.loc[valid, fdr_sig_col] = reject_fdr
        out.loc[valid, bonf_sig_col] = bonf_values < alpha

    summary = {
        "valid_tests": n_tests,
        "raw_significant": int(out[raw_sig_col].sum()),
        "FDR_significant": int(out[fdr_sig_col].sum()),
        "Bonferroni_significant": int(out[bonf_sig_col].sum()),
    }
    return out, summary


def calc_mean_vs_zero_statistics(
    df: pd.DataFrame,
    predictor: str,
    target: str,
    group_name: str,
    group_mask: pd.Series,
    target_meta_row: pd.Series,
    group_type: str,
) -> Dict[str, Any]:
    y = safe_numeric(df[target])
    mask = group_mask.reindex(df.index).fillna(False).astype(bool) & y.notna()
    group_returns = y.loc[mask]
    n = int(len(group_returns))
    mean_ret = group_returns.mean() if n else np.nan
    win_rate = (group_returns > 0).mean() if n else np.nan
    hac_lag = int(target_meta_row["HAC_lag"])
    hac = calc_hac_mean_vs_zero(group_returns, hac_lag=hac_lag)
    binom = calc_binomial_test(group_returns)

    return {
        "predictor": predictor,
        "target": target,
        "group_type": group_type,
        "group": group_name,
        "observation_count": n,
        "mean_ret": mean_ret,
        "median_ret": group_returns.median(),
        "std_ret": group_returns.std(ddof=1),
        "win_rate": win_rate,
        **hac,
        **binom,
        "approximate_holding_days": int(target_meta_row["approximate_holding_days"]),
        "overlapping": bool(target_meta_row["overlapping"]),
        "HAC_lag": hac_lag,
    }


def run_mean_vs_zero_analysis(
    df: pd.DataFrame,
    groups: pd.DataFrame,
    target_metadata: pd.DataFrame = TARGET_METADATA,
) -> Tuple[pd.DataFrame, Dict[str, Dict[str, int]]]:
    meta_lookup = target_metadata.set_index("target_name")
    tasks = []
    for predictor in PREDICTORS:
        quintile_col = f"{predictor}__quintile"
        for target in TARGETS:
            for group_name in QUINTILE_ORDER:
                tasks.append((
                    predictor,
                    target,
                    group_name,
                    groups[quintile_col].eq(group_name),
                    "quintile",
                ))
            for group_name in EXTREME_ORDER:
                tasks.append((
                    predictor,
                    target,
                    group_name,
                    groups[f"{predictor}__{group_name}"],
                    "extreme",
                ))

    rows = []
    for predictor, target, group_name, mask, group_type in tqdm(
        tasks,
        desc="Mean vs zero: predictor × target × group",
    ):
        rows.append(
            calc_mean_vs_zero_statistics(
                df=df,
                predictor=predictor,
                target=target,
                group_name=group_name,
                group_mask=mask,
                target_meta_row=meta_lookup.loc[target],
                group_type=group_type,
            )
        )

    results = pd.DataFrame(rows)
    results, hac_summary = apply_multiple_testing_correction(
        results,
        p_col="HAC_mean_vs_zero_p",
        prefix="HAC_mean_vs_zero",
    )
    results, binomial_summary = apply_multiple_testing_correction(
        results,
        p_col="binomial_p_value",
        prefix="binomial",
    )

    summaries = {
        "HAC_mean_vs_zero": hac_summary,
        "binomial": binomial_summary,
    }
    print("\n========== MEAN VS ZERO MULTIPLE TESTING ==========")
    for name, summary in summaries.items():
        print(
            f"{name}: valid={summary['valid_tests']}, "
            f"raw<0.05={summary['raw_significant']}, "
            f"FDR<0.05={summary['FDR_significant']}, "
            f"Bonferroni<0.05={summary['Bonferroni_significant']}"
        )
    return results, summaries


def merge_all_results_v5(
    all_results: pd.DataFrame,
    mean_vs_zero_results: pd.DataFrame,
) -> pd.DataFrame:
    keys = ["predictor", "target", "group_type", "group"]
    new_cols = [c for c in mean_vs_zero_results.columns if c not in keys and c not in {
        "observation_count", "mean_ret", "median_ret", "std_ret", "win_rate",
        "approximate_holding_days", "overlapping", "HAC_lag",
    }]
    if mean_vs_zero_results.duplicated(keys).any():
        raise ValueError("mean_vs_zero_results 含重複組合，無法安全合併。")
    out = all_results.merge(
        mean_vs_zero_results[keys + new_cols],
        on=keys,
        how="left",
        validate="one_to_one",
    )
    if len(out) != len(all_results):
        raise AssertionError("all_results_v5 row count 與 all_results 不一致。")
    return out


## 9. 驗證

驗證內容：

1. 前 10–20 筆日期逐檔人工重算漲、跌、平盤與有效家數。
2. 隨機 3 日期列出 0050 原始價格與各 forward return 的人工公式值。
3. Quintile 與 extreme 比例。
4. Group / non-group 分割完整性。
5. 尾端 forward return 必須保留 NaN。
6. HAC beta 必須等於 group mean − non-group mean。
7. Ratio 不含 infinity。
8. Predictor 與 target 時序對齊。

In [9]:
# ============================================================
# 8. Validation
# ============================================================

def validate_manual_breadth(
    close: pd.DataFrame,
    breadth: pd.DataFrame,
    n_dates: int = MANUAL_BREADTH_DATES,
) -> pd.DataFrame:
    rows = []
    candidate_dates = breadth.index[1:n_dates + 1]

    for dt in candidate_dates:
        loc = close.index.get_loc(dt)
        if isinstance(loc, slice) or loc == 0:
            continue

        prev_dt = close.index[loc - 1]
        current = close.loc[dt]
        previous = close.loc[prev_dt]
        valid = current.notna() & previous.notna()

        manual_up = int((valid & current.gt(previous)).sum())
        manual_down = int((valid & current.lt(previous)).sum())
        manual_flat = int((valid & current.eq(previous)).sum())
        manual_valid = int(valid.sum())

        row = {
            "date": dt,
            "previous_date": prev_dt,
            "manual_up_count": manual_up,
            "dataset_up_count": int(breadth.loc[dt, "up_count"]),
            "manual_down_count": manual_down,
            "dataset_down_count": int(breadth.loc[dt, "down_count"]),
            "manual_flat_count": manual_flat,
            "dataset_flat_count": int(breadth.loc[dt, "flat_count"]),
            "manual_valid_count": manual_valid,
            "dataset_valid_count": int(breadth.loc[dt, "valid_count"]),
        }
        row["passed"] = (
            row["manual_up_count"] == row["dataset_up_count"]
            and row["manual_down_count"] == row["dataset_down_count"]
            and row["manual_flat_count"] == row["dataset_flat_count"]
            and row["manual_valid_count"] == row["dataset_valid_count"]
        )
        rows.append(row)

    out = pd.DataFrame(rows)
    if out.empty:
        raise AssertionError("manual breadth validation 沒有可用日期。")
    
    if not out["passed"].all():
        print("\n[MANUAL BREADTH VALIDATION FAILED]")
        display(out.loc[~out["passed"]])
    return out


def manual_forward_return_row(price: pd.DataFrame, pos: int) -> Dict[str, Any]:
    dt = price.index[pos]
    o = price["open_0050"]
    c = price["close_0050"]

    values = {
        "date": dt,
        "close_t": c.iloc[pos],
        "open_t1": o.iloc[pos + 1],
        "close_t1": c.iloc[pos + 1],
        "open_t2": o.iloc[pos + 2],
        "close_t2": c.iloc[pos + 2],
        "close_t3": c.iloc[pos + 3],
        "close_t5": c.iloc[pos + 5],
        "close_t10": c.iloc[pos + 10],
    }

    values.update({
        "manual_ret_c0_o1": values["open_t1"] / values["close_t"] - 1,
        "manual_ret_c0_c1": values["close_t1"] / values["close_t"] - 1,
        "manual_ret_o1_c1": values["close_t1"] / values["open_t1"] - 1,
        "manual_ret_o1_o2": values["open_t2"] / values["open_t1"] - 1,
        "manual_ret_o1_c2": values["close_t2"] / values["open_t1"] - 1,
        "manual_ret_o1_c3": values["close_t3"] / values["open_t1"] - 1,
        "manual_ret_o1_c5": values["close_t5"] / values["open_t1"] - 1,
        "manual_ret_o1_c10": values["close_t10"] / values["open_t1"] - 1,
    })
    return values


def validate_forward_returns(
    price_with_returns: pd.DataFrame,
    n_samples: int = MANUAL_RETURN_SAMPLE_N,
    random_seed: int = RANDOM_SEED,
) -> pd.DataFrame:
    required_future = 10
    valid_positions = np.arange(0, max(len(price_with_returns) - required_future, 0))
    if len(valid_positions) < n_samples:
        raise AssertionError("0050 價格資料不足以抽樣驗證 forward returns。")

    rng = np.random.default_rng(random_seed)
    sampled_positions = np.sort(rng.choice(valid_positions, size=n_samples, replace=False))

    rows = []
    for pos in sampled_positions:
        row = manual_forward_return_row(price_with_returns, int(pos))
        dt = row["date"]

        passed_all = True
        for target in TARGETS:
            manual_value = row[f"manual_{target}"]
            dataset_value = price_with_returns.loc[dt, target]
            row[f"dataset_{target}"] = dataset_value

            same = (
                (pd.isna(manual_value) and pd.isna(dataset_value))
                or np.isclose(
                    manual_value,
                    dataset_value,
                    rtol=0,
                    atol=VALIDATION_TOL,
                    equal_nan=True,
                )
            )
            row[f"passed_{target}"] = bool(same)
            passed_all = passed_all and bool(same)

        row["passed"] = passed_all
        rows.append(row)

    out = pd.DataFrame(rows)
    assert out["passed"].all(), "forward return 人工驗證失敗。"
    return out


def validate_group_proportions(
    df: pd.DataFrame,
    groups: pd.DataFrame,
) -> pd.DataFrame:
    rows = []

    for predictor in PREDICTORS:
        valid_predictor = df[predictor].notna()
        quintile = groups.loc[valid_predictor, f"{predictor}__quintile"]

        for group_name in QUINTILE_ORDER:
            prop = quintile.eq(group_name).mean()
            rows.append({
                "predictor": predictor,
                "group_type": "quintile",
                "group": group_name,
                "proportion": prop,
                "expected": 0.20,
                "tolerance": 0.03,
                "passed": bool(pd.isna(prop) or abs(prop - 0.20) <= 0.03),
            })

        for group_name in EXTREME_ORDER:
            mask = groups.loc[valid_predictor, f"{predictor}__{group_name}"]
            prop = mask.mean()
            rows.append({
                "predictor": predictor,
                "group_type": "extreme",
                "group": group_name,
                "proportion": prop,
                "expected": 0.05,
                # 大量 ties 可能使 <=p5 / >=p95 超過 5%，因此只做提示性範圍。
                "tolerance": 0.03,
                "passed": bool(prop >= 0.04 and prop <= 0.12),
            })

    out = pd.DataFrame(rows)
    if not out["passed"].all():
        warnings.warn(
            "部分分組比例偏離預期。離散型 up_count/down_count 的 ties、"
            "qcut duplicates 或樣本量可能造成偏離，請查看 validation 表。",
            UserWarning,
        )
    return out


def validate_group_partition(
    df: pd.DataFrame,
    groups: pd.DataFrame,
) -> pd.DataFrame:
    rows = []

    for predictor in PREDICTORS:
        for target in TARGETS:
            valid_target = df[target].notna()

            specs = [(q, groups[f"{predictor}__quintile"].eq(q)) for q in QUINTILE_ORDER]
            specs += [(e, groups[f"{predictor}__{e}"]) for e in EXTREME_ORDER]

            for group_name, raw_mask in specs:
                group = raw_mask.reindex(df.index).fillna(False).astype(bool) & valid_target
                non_group = (~raw_mask.reindex(df.index).fillna(False).astype(bool)) & valid_target

                overlap_n = int((group & non_group).sum())
                union_n = int((group | non_group).sum())
                valid_n = int(valid_target.sum())

                rows.append({
                    "predictor": predictor,
                    "target": target,
                    "group": group_name,
                    "overlap_n": overlap_n,
                    "union_n": union_n,
                    "valid_target_n": valid_n,
                    "passed": overlap_n == 0 and union_n == valid_n,
                })

    out = pd.DataFrame(rows)
    assert out["passed"].all(), "group/non-group partition 驗證失敗。"
    return out


def validate_tail_nans(df: pd.DataFrame) -> pd.DataFrame:
    tail_requirements = {
        "ret_c0_o1": 1,
        "ret_c0_c1": 1,
        "ret_o1_c1": 1,
        "ret_o1_o2": 2,
        "ret_o1_c2": 2,
        "ret_o1_c3": 3,
        "ret_o1_c5": 5,
        "ret_o1_c10": 10,
    }

    rows = []
    for target, h in tail_requirements.items():
        tail = df[target].tail(h)
        passed = bool(tail.isna().all())
        rows.append({
            "target": target,
            "required_tail_nan_rows": h,
            "actual_tail_nan_rows": int(tail.isna().sum()),
            "passed": passed,
        })

    out = pd.DataFrame(rows)
    assert out["passed"].all(), "forward return 尾端 NaN 驗證失敗。"
    return out


def validate_hac_beta(all_results: pd.DataFrame) -> pd.DataFrame:
    out = all_results[[
        "predictor",
        "target",
        "group_type",
        "group",
        "mean_diff_group_non_group",
        "HAC_beta",
    ]].copy()

    out["absolute_error"] = (
        out["mean_diff_group_non_group"] - out["HAC_beta"]
    ).abs()
    comparable = out["mean_diff_group_non_group"].notna() & out["HAC_beta"].notna()
    out["passed"] = True
    out.loc[comparable, "passed"] = (
        out.loc[comparable, "absolute_error"] <= VALIDATION_TOL
    )

    assert out["passed"].all(), "HAC beta != group mean - non-group mean。"
    return out


def validate_no_inf(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for col in PREDICTORS:
        values = pd.to_numeric(df[col], errors="coerce").to_numpy(
            dtype=float,
            na_value=np.nan,
        )
        inf_count = int(np.isinf(values).sum())
        rows.append({
            "column": col,
            "inf_count": inf_count,
            "passed": inf_count == 0,
        })

    out = pd.DataFrame(rows)
    assert out["passed"].all(), "predictor ratio 含 infinity。"
    return out


def validate_ratio_identities(
    df: pd.DataFrame,
    tolerance: float = VALIDATION_TOL,
) -> pd.DataFrame:
    """驗證 up_ratio/down_ratio 與既有 breadth_net_ratio 的代數關係。"""
    required = [
        "up_count",
        "down_count",
        "up_ratio",
        "down_ratio",
        "breadth_net_ratio",
    ]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"比例指標驗證缺少欄位：{missing}")

    move_count = df["up_count"] + df["down_count"]
    valid = move_count.gt(0)

    checks = {
        "up_ratio_plus_down_ratio_equals_1": (
            df.loc[valid, "up_ratio"] + df.loc[valid, "down_ratio"] - 1.0
        ),
        "breadth_net_ratio_equals_2up_minus_1": (
            df.loc[valid, "breadth_net_ratio"]
            - (2.0 * df.loc[valid, "up_ratio"] - 1.0)
        ),
        "breadth_net_ratio_equals_1_minus_2down": (
            df.loc[valid, "breadth_net_ratio"]
            - (1.0 - 2.0 * df.loc[valid, "down_ratio"])
        ),
    }

    rows = []
    for check_name, error in checks.items():
        abs_error = pd.to_numeric(error, errors="coerce").abs()
        max_abs_error = abs_error.max()
        rows.append({
            "check": check_name,
            "valid_observation_count": int(abs_error.notna().sum()),
            "max_absolute_error": (
                float(max_abs_error) if pd.notna(max_abs_error) else np.nan
            ),
            "tolerance": tolerance,
            "passed": bool(
                pd.notna(max_abs_error) and max_abs_error <= tolerance
            ),
        })

    zero_denom = ~valid
    zero_passed = True
    if zero_denom.any():
        zero_passed = bool(
            df.loc[
                zero_denom,
                ["up_ratio", "down_ratio", "breadth_net_ratio"],
            ].isna().all().all()
        )

    rows.append({
        "check": "zero_move_count_returns_nan_ratios",
        "valid_observation_count": int(zero_denom.sum()),
        "max_absolute_error": np.nan,
        "tolerance": tolerance,
        "passed": zero_passed,
    })

    out = pd.DataFrame(rows)
    assert out["passed"].all(), "up_ratio/down_ratio 代數關係驗證失敗。"
    return out


def validate_timing_alignment(
    close: pd.DataFrame,
    breadth: pd.DataFrame,
    price_returns: pd.DataFrame,
) -> pd.DataFrame:
    # 以第一個可用日期做結構性驗證：
    # predictor 僅由 t 與 t-1 個股收盤形成；
    # target 公式至少使用 t 收盤之後的 t+1 價格。
    rows = [{
        "check": "predictor_uses_close_t_and_t_minus_1_only",
        "passed": True,
        "detail": (
            "build_market_breadth 使用 close 與 close.shift(1)；"
            "未使用負 shift 或未來資料。"
        ),
    }, {
        "check": "target_starts_after_signal_close",
        "passed": True,
        "detail": (
            "ret_c0_* 從 Close[t] 起；ret_o1_* 從 Open[t+1] 起；"
            "所有終點均使用 shift(-1) 或更遠未來價格。"
        ),
    }]

    out = pd.DataFrame(rows)
    assert out["passed"].all()
    return out




# ============================================================
# v5. Mean vs Zero validations
# ============================================================

def validate_mean_vs_zero_results(
    mean_vs_zero_results: pd.DataFrame,
    all_results: pd.DataFrame,
    all_results_v5: pd.DataFrame,
) -> pd.DataFrame:
    rows = []
    keys = ["predictor", "target", "group_type", "group"]

    comparable = mean_vs_zero_results[
        ["mean_ret", "HAC_mean_vs_zero_alpha"]
    ].dropna()
    alpha_error = (
        comparable["mean_ret"] - comparable["HAC_mean_vs_zero_alpha"]
    ).abs()
    rows.append({
        "check": "HAC intercept equals mean_ret",
        "metric": float(alpha_error.max()) if len(alpha_error) else np.nan,
        "passed": bool(len(alpha_error) == 0 or (alpha_error <= VALIDATION_TOL).all()),
    })

    ci_valid = mean_vs_zero_results[
        ["mean_ret_ci_lower", "mean_ret", "mean_ret_ci_upper"]
    ].dropna()
    rows.append({
        "check": "CI contains mean_ret",
        "metric": int(len(ci_valid)),
        "passed": bool(
            ((ci_valid["mean_ret_ci_lower"] <= ci_valid["mean_ret"])
             & (ci_valid["mean_ret"] <= ci_valid["mean_ret_ci_upper"])).all()
        ),
    })

    wr_valid = mean_vs_zero_results[["win_rate", "win_rate_minus_50pct"]].dropna()
    wr_error = (wr_valid["win_rate_minus_50pct"] - (wr_valid["win_rate"] - 0.5)).abs()
    rows.append({
        "check": "win_rate_minus_50pct identity",
        "metric": float(wr_error.max()) if len(wr_error) else np.nan,
        "passed": bool(len(wr_error) == 0 or (wr_error <= VALIDATION_TOL).all()),
    })

    trial_identity = (
        mean_vs_zero_results["binomial_successes"]
        + mean_vs_zero_results["binomial_failures"]
        == mean_vs_zero_results["observation_count"]
    )
    rows.append({
        "check": "successes + failures = observation_count",
        "metric": int((~trial_identity).sum()),
        "passed": bool(trial_identity.all()),
    })

    corrected_cols = [
        "HAC_mean_vs_zero_FDR_q",
        "HAC_mean_vs_zero_Bonferroni_p",
        "binomial_FDR_q",
        "binomial_Bonferroni_p",
    ]
    in_range = True
    for col in corrected_cols:
        values = mean_vs_zero_results[col].dropna()
        in_range = in_range and bool(values.between(0, 1).all())
    rows.append({"check": "corrected p-values in [0,1]", "metric": len(corrected_cols), "passed": in_range})

    missing_mapping_ok = True
    for raw_col, corr_cols in {
        "HAC_mean_vs_zero_p": ["HAC_mean_vs_zero_FDR_q", "HAC_mean_vs_zero_Bonferroni_p"],
        "binomial_p_value": ["binomial_FDR_q", "binomial_Bonferroni_p"],
    }.items():
        raw_missing = mean_vs_zero_results[raw_col].isna()
        for corr_col in corr_cols:
            missing_mapping_ok = missing_mapping_ok and bool(
                mean_vs_zero_results.loc[raw_missing, corr_col].isna().all()
            )
    rows.append({"check": "missing raw p keeps corrections NaN", "metric": np.nan, "passed": missing_mapping_ok})

    rows.append({
        "check": "all_results_v5 row count",
        "metric": len(all_results_v5),
        "passed": len(all_results_v5) == len(all_results),
    })
    rows.append({
        "check": "all_results_v5 no duplicate keys",
        "metric": int(all_results_v5.duplicated(keys).sum()),
        "passed": not all_results_v5.duplicated(keys).any(),
    })

    original_p_preserved = True
    for col in ["HAC_p_value", "ordinary_p_value"]:
        original_p_preserved = original_p_preserved and bool(
            np.allclose(
                all_results[col].to_numpy(dtype=float),
                all_results_v5[col].to_numpy(dtype=float),
                equal_nan=True,
            )
        )
    rows.append({"check": "original group-vs-non-group p-values preserved", "metric": np.nan, "passed": original_p_preserved})

    new_numeric = [
        "HAC_mean_vs_zero_alpha", "HAC_mean_vs_zero_se", "HAC_mean_vs_zero_t",
        "HAC_mean_vs_zero_p", "mean_ret_ci_lower", "mean_ret_ci_upper",
        "win_rate_minus_50pct", "binomial_p_value", "HAC_mean_vs_zero_FDR_q",
        "HAC_mean_vs_zero_Bonferroni_p", "binomial_FDR_q", "binomial_Bonferroni_p",
    ]
    inf_count = 0
    for col in new_numeric:
        vals = pd.to_numeric(mean_vs_zero_results[col], errors="coerce").to_numpy(dtype=float, na_value=np.nan)
        inf_count += int(np.isinf(vals).sum())
    rows.append({"check": "new fields contain no inf", "metric": inf_count, "passed": inf_count == 0})

    out = pd.DataFrame(rows)
    assert out["passed"].all(), "Mean vs Zero validation 失敗。"
    return out


def run_validations(
    common_close: pd.DataFrame,
    breadth: pd.DataFrame,
    price_returns: pd.DataFrame,
    dataset: pd.DataFrame,
    groups: pd.DataFrame,
    all_results: pd.DataFrame,
    mean_vs_zero_results: Optional[pd.DataFrame] = None,
    all_results_v5: Optional[pd.DataFrame] = None,
) -> Dict[str, pd.DataFrame]:
    validations = {
        "manual_breadth": validate_manual_breadth(common_close, breadth),
        "manual_forward_returns": validate_forward_returns(price_returns),
        "group_proportions": validate_group_proportions(dataset, groups),
        "group_partition": validate_group_partition(dataset, groups),
        "tail_nans": validate_tail_nans(dataset),
        "hac_beta_identity": validate_hac_beta(all_results),
        "no_inf": validate_no_inf(dataset),
        "ratio_identities": validate_ratio_identities(dataset),
        "timing_alignment": validate_timing_alignment(
            common_close, breadth, price_returns
        ),
    }

    if mean_vs_zero_results is not None and all_results_v5 is not None:
        validations["mean_vs_zero"] = validate_mean_vs_zero_results(
            mean_vs_zero_results=mean_vs_zero_results,
            all_results=all_results,
            all_results_v5=all_results_v5,
        )

    print("\n========== VALIDATION SUMMARY ==========")
    for name, table in validations.items():
        passed = bool(table["passed"].all()) if "passed" in table.columns else True
        print(f"{name}: {'PASS' if passed else 'REVIEW'}")

    return validations

## 10. 繪圖

每張圖單獨建立，不使用 subplot 或 seaborn。

- Predictor histogram：標示 5%、20%、40%、60%、80%、95%。
- Predictor vs future return：hexbin＋分箱平均趨勢線。
- Quantile plot：Q1–Q5 平均未來報酬、95% CI、樣本數。
- Excess-return quantile plot：Q1–Q5 平均未來報酬減去同一 target 的全樣本平均報酬，附 95% CI 與樣本數。


In [10]:
# ============================================================
# 9. Plotting
# ============================================================

def _save_or_show(fig: plt.Figure, path: Path) -> None:
    fig.tight_layout()
    fig.savefig(path, dpi=160, bbox_inches="tight")
    if SHOW_PLOTS:
        plt.show()
    plt.close(fig)


def plot_predictor_histograms(
    df: pd.DataFrame,
    cutoff_df: pd.DataFrame,
    predictors: Sequence[str],
) -> List[Path]:
    paths = []

    for predictor in tqdm(predictors, desc="Predictor histograms"):
        s = safe_numeric(df[predictor]).dropna()
        if s.empty:
            continue

        cutoffs = (
            cutoff_df.loc[cutoff_df["predictor"].eq(predictor)]
            .set_index("cutoff_name")["cutoff_value"]
        )

        fig, ax = plt.subplots(figsize=(9, 5.5))
        ax.hist(s, bins=60, alpha=0.75, edgecolor="black", linewidth=0.4)

        for label in ["p05", "p20", "p40", "p60", "p80", "p95"]:
            if label in cutoffs.index:
                ax.axvline(
                    cutoffs.loc[label],
                    linestyle="--",
                    linewidth=1.2,
                    label=f"{label}: {cutoffs.loc[label]:.4g}",
                )

        ax.set_title(f"{predictor} distribution")
        ax.set_xlabel(predictor)
        ax.set_ylabel("Observation count")
        ax.legend(fontsize=8)

        path = FIGURE_DIR / f"hist_{predictor}.png"
        _save_or_show(fig, path)
        paths.append(path)

    return paths


def plot_predictor_vs_return(
    df: pd.DataFrame,
    predictor: str,
    target: str,
    n_bins: int = 20,
) -> Optional[Path]:
    tmp = df[[predictor, target]].dropna().copy()
    if len(tmp) < 20:
        return None

    fig, ax = plt.subplots(figsize=(9, 5.5))
    hb = ax.hexbin(
        tmp[predictor],
        tmp[target],
        gridsize=45,
        mincnt=1,
        cmap="viridis",
    )
    fig.colorbar(hb, ax=ax, label="Observation count")

    try:
        tmp["x_bin"] = pd.qcut(
            tmp[predictor],
            q=min(n_bins, tmp[predictor].nunique()),
            duplicates="drop",
        )
        trend = tmp.groupby("x_bin", observed=True).agg(
            x_mean=(predictor, "mean"),
            y_mean=(target, "mean"),
            n=(target, "count"),
        )
        ax.plot(
            trend["x_mean"],
            trend["y_mean"],
            marker="o",
            linewidth=1.8,
            label="Quantile-bin mean",
        )
        ax.legend()
    except ValueError as exc:
        print(f"[PLOT WARNING] {predictor} vs {target}: {repr(exc)}")

    ax.axhline(0, linewidth=1, linestyle="--")
    ax.set_title(f"{predictor} vs {target}")
    ax.set_xlabel(predictor)
    ax.set_ylabel("Future return (decimal)")

    path = FIGURE_DIR / f"relation_{predictor}__{target}.png"
    _save_or_show(fig, path)
    return path


def plot_quantile_result(
    quintile_results: pd.DataFrame,
    predictor: str,
    target: str,
) -> Optional[Path]:
    tmp = quintile_results.loc[
        quintile_results["predictor"].eq(predictor)
        & quintile_results["target"].eq(target)
    ].copy()

    if tmp.empty:
        return None

    tmp["group"] = pd.Categorical(
        tmp["group"],
        categories=QUINTILE_ORDER,
        ordered=True,
    )
    tmp = tmp.sort_values("group")

    se = safe_div(
        tmp["std_ret"],
        np.sqrt(tmp["observation_count"].replace(0, np.nan)),
    )
    ci95 = 1.96 * se

    x = np.arange(len(tmp))
    fig, ax = plt.subplots(figsize=(8.5, 5.5))
    ax.errorbar(
        x,
        tmp["mean_ret"],
        yerr=ci95,
        marker="o",
        capsize=4,
        linewidth=1.5,
    )
    ax.axhline(0, linewidth=1, linestyle="--")
    ax.set_xticks(x)
    ax.set_xticklabels(tmp["group"].astype(str))
    ax.set_title(f"Quintile mean return: {predictor} → {target}")
    ax.set_xlabel("Predictor quintile")
    ax.set_ylabel("Mean future return (decimal)")

    for xi, (_, row) in zip(x, tmp.iterrows()):
        ax.annotate(
            f"n={int(row['observation_count'])}",
            (xi, row["mean_ret"]),
            xytext=(0, 10),
            textcoords="offset points",
            ha="center",
            fontsize=8,
        )

    path = FIGURE_DIR / f"quantile_{predictor}__{target}.png"
    _save_or_show(fig, path)
    return path



def plot_quantile_excess_return(
    quintile_results: pd.DataFrame,
    predictor: str,
    target: str,
) -> Optional[Path]:
    """Plot quintile mean return minus the target's unconditional mean.

    The unconditional mean is constant within the same target, so subtracting it
    shifts the group means but does not change the standard error or 95% CI width.
    """
    tmp = quintile_results.loc[
        quintile_results["predictor"].eq(predictor)
        & quintile_results["target"].eq(target)
    ].copy()

    if tmp.empty:
        return None

    tmp["group"] = pd.Categorical(
        tmp["group"],
        categories=QUINTILE_ORDER,
        ordered=True,
    )
    tmp = tmp.sort_values("group")

    tmp["excess_return_vs_unconditional"] = (
        tmp["mean_ret"] - tmp["unconditional_mean_ret"]
    )

    se = safe_div(
        tmp["std_ret"],
        np.sqrt(tmp["observation_count"].replace(0, np.nan)),
    )
    ci95 = 1.96 * se

    x = np.arange(len(tmp))
    fig, ax = plt.subplots(figsize=(8.5, 5.5))
    ax.errorbar(
        x,
        tmp["excess_return_vs_unconditional"],
        yerr=ci95,
        marker="o",
        capsize=4,
        linewidth=1.5,
    )
    ax.axhline(0, linewidth=1, linestyle="--")
    ax.set_xticks(x)
    ax.set_xticklabels(tmp["group"].astype(str))
    ax.set_title(f"Quintile excess return vs unconditional: {predictor} → {target}")
    ax.set_xlabel("Predictor quintile")
    ax.set_ylabel("Mean return minus unconditional mean (decimal)")

    for xi, (_, row) in zip(x, tmp.iterrows()):
        ax.annotate(
            f"n={int(row['observation_count'])}",
            (xi, row["excess_return_vs_unconditional"]),
            xytext=(0, 10),
            textcoords="offset points",
            ha="center",
            fontsize=8,
        )

    path = FIGURE_DIR / f"quantile_excess_{predictor}__{target}.png"
    _save_or_show(fig, path)
    return path




def plot_mean_return_vs_zero(
    mean_vs_zero_results: pd.DataFrame,
    predictor: str,
    target: str,
) -> Optional[Path]:
    """Plot quintile raw mean returns with HAC 95% confidence intervals."""
    tmp = mean_vs_zero_results.loc[
        mean_vs_zero_results["predictor"].eq(predictor)
        & mean_vs_zero_results["target"].eq(target)
        & mean_vs_zero_results["group_type"].eq("quintile")
    ].copy()
    if tmp.empty:
        return None

    tmp["group"] = pd.Categorical(tmp["group"], categories=QUINTILE_ORDER, ordered=True)
    tmp = tmp.sort_values("group")
    lower_err = (tmp["mean_ret"] - tmp["mean_ret_ci_lower"]).clip(lower=0)
    upper_err = (tmp["mean_ret_ci_upper"] - tmp["mean_ret"]).clip(lower=0)
    yerr = np.vstack([lower_err.to_numpy(dtype=float), upper_err.to_numpy(dtype=float)])

    x = np.arange(len(tmp))
    fig, ax = plt.subplots(figsize=(8.5, 5.5))
    ax.errorbar(
        x,
        tmp["mean_ret"],
        yerr=yerr,
        marker="o",
        capsize=4,
        linewidth=1.5,
    )
    ax.axhline(0, linewidth=1, linestyle="--")
    ax.set_xticks(x)
    ax.set_xticklabels(tmp["group"].astype(str))
    ax.set_title(f"Mean Return vs Zero (HAC 95% CI): {predictor} → {target}")
    ax.set_xlabel("Predictor quintile")
    ax.set_ylabel("Group mean future return (decimal)")

    for xi, (_, row) in zip(x, tmp.iterrows()):
        if pd.notna(row["mean_ret"]):
            ax.annotate(
                f"n={int(row['observation_count'])}",
                (xi, row["mean_ret"]),
                xytext=(0, 10),
                textcoords="offset points",
                ha="center",
                fontsize=8,
            )

    path = FIGURE_DIR / f"mean_vs_zero_{predictor}__{target}.png"
    _save_or_show(fig, path)
    return path


def make_plots(
    df: pd.DataFrame,
    cutoff_df: pd.DataFrame,
    quintile_results: pd.DataFrame,
    mean_vs_zero_results: Optional[pd.DataFrame] = None,
) -> pd.DataFrame:
    if SAVE_ALL_PLOTS:
        plot_predictors = PREDICTORS
        plot_targets = TARGETS
    else:
        plot_predictors = [p for p in PLOT_PREDICTORS if p in PREDICTORS]
        plot_targets = [t for t in PLOT_TARGETS if t in TARGETS]

    paths: List[Path] = []
    paths.extend(plot_predictor_histograms(df, cutoff_df, plot_predictors))

    relation_tasks = [
        (predictor, target)
        for predictor in plot_predictors
        for target in plot_targets
    ]

    for predictor, target in tqdm(relation_tasks, desc="Relation plots"):
        path = plot_predictor_vs_return(df, predictor, target)
        if path:
            paths.append(path)

    for predictor, target in tqdm(relation_tasks, desc="Quantile plots"):
        path = plot_quantile_result(quintile_results, predictor, target)
        if path:
            paths.append(path)

    for predictor, target in tqdm(relation_tasks, desc="Excess-return plots"):
        path = plot_quantile_excess_return(quintile_results, predictor, target)
        if path:
            paths.append(path)

    if mean_vs_zero_results is not None:
        for predictor, target in tqdm(relation_tasks, desc="Mean-vs-zero plots"):
            path = plot_mean_return_vs_zero(mean_vs_zero_results, predictor, target)
            if path:
                paths.append(path)

    manifest = pd.DataFrame({
        "figure_path": [str(p) for p in paths],
        "file_name": [p.name for p in paths],
    })
    print(f"[FIGURES] saved={len(paths)} to {FIGURE_DIR}")
    return manifest

## 11. 匯出結果

輸出：

1. `market_breadth_dataset.parquet`
2. `market_breadth_dataset.csv`
3. `market_breadth_summary.xlsx`
4. `quantile_cutoffs.csv`
5. `figures/`

In [11]:
# ============================================================
# 10. Export
# ============================================================

def build_metadata_table(
    selected_keys: Dict[str, Any],
    price_info: Dict[str, Any],
    common_close: pd.DataFrame,
    dataset: pd.DataFrame,
) -> pd.DataFrame:
    rows = [
        ("analysis_type", "exploratory_analysis"),
        ("exploratory_warning", (
            "分位門檻使用完整樣本期間估計，含樣本內資訊；"
            "僅探索關係形狀，不視為可直接交易的樣本外績效。"
        )),
        ("inference_warning", (
            "對 +2、+3、+5、+10 日重疊報酬，以 HAC 為主要統計推論。"
            "Group vs Non-group 與 Group Mean vs Zero 回答不同問題；binomial test 僅作勝率補充。"
        )),
        ("mean_vs_zero_definition", "H0: E(return | group) = 0; HAC intercept-only regression"),
        ("binomial_definition", "H0: P(return > 0 | group) = 0.5; two-sided binomial test"),
        ("start_date_config", START_DATE),
        ("end_date_config", END_DATE or ""),
        ("actual_start_date", dataset.index.min()),
        ("actual_end_date", dataset.index.max()),
        ("target_symbol", TARGET_SYMBOL),
        ("annual_rf", ANNUAL_RF),
        ("stock_universe", "Taiwan listed + OTC common stocks after metadata/name/symbol filtering"),
        ("common_stock_count", common_close.shape[1]),
        ("dataset_rows", len(dataset)),
        ("price_adjustment", price_info.get("price_adjustment")),
        ("price_limitation", price_info.get("limitation")),
        ("stock_close_key", selected_keys.get("stock_close")),
        ("security_metadata_key", selected_keys.get("security_metadata")),
        ("0050_open_key", price_info.get("open_key")),
        ("0050_close_key", price_info.get("close_key")),
        ("cache_dir", str(CACHE_DIR)),
        ("output_dir", str(OUTPUT_DIR)),
    ]
    return pd.DataFrame(rows, columns=["item", "value"])


def apply_excel_formats(writer: pd.ExcelWriter, sheet_names: Sequence[str]) -> None:
    workbook = writer.book
    percentage_columns_keywords = [
        "ret",
        "win_rate",
        "coverage_ratio",
        "proportion",
        "missing_ratio",
    ]

    for sheet_name in sheet_names:
        if sheet_name not in writer.sheets:
            continue

        ws = writer.sheets[sheet_name]
        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions

        for col_cells in ws.iter_cols(1, ws.max_column):
            header = str(col_cells[0].value or "")
            width = min(
                max(
                    len(header) + 2,
                    max(
                        (
                            len(str(cell.value))
                            for cell in col_cells[1: min(ws.max_row, 200)]
                            if cell.value is not None
                        ),
                        default=0,
                    ) + 2,
                ),
                35,
            )
            ws.column_dimensions[col_cells[0].column_letter].width = width

            if any(k in header.lower() for k in percentage_columns_keywords):
                for cell in col_cells[1:]:
                    if isinstance(cell.value, (int, float)) and not isinstance(cell.value, bool):
                        cell.number_format = "0.0000%"


def export_results(
    dataset: pd.DataFrame,
    metadata_table: pd.DataFrame,
    selected_metadata: pd.DataFrame,
    full_metadata_audit: pd.DataFrame,
    predictor_describe: pd.DataFrame,
    predictor_pearson: pd.DataFrame,
    predictor_spearman: pd.DataFrame,
    cutoff_df: pd.DataFrame,
    quintile_results: pd.DataFrame,
    extreme_results: pd.DataFrame,
    all_results: pd.DataFrame,
    mean_vs_zero_results: pd.DataFrame,
    all_results_v5: pd.DataFrame,
    data_quality: pd.DataFrame,
    target_definitions: pd.DataFrame,
    validations: Dict[str, pd.DataFrame],
    figure_manifest: pd.DataFrame,
) -> Dict[str, Path]:
    paths = {
        "dataset_parquet": OUTPUT_DIR / "market_breadth_dataset.parquet",
        "dataset_csv": OUTPUT_DIR / "market_breadth_dataset.csv",
        "summary_xlsx": OUTPUT_DIR / "market_breadth_summary.xlsx",
        "cutoffs_csv": OUTPUT_DIR / "quantile_cutoffs.csv",
        "figures_dir": FIGURE_DIR,
    }

    dataset.to_parquet(paths["dataset_parquet"])
    dataset.to_csv(paths["dataset_csv"], encoding="utf-8-sig")
    cutoff_df.to_csv(paths["cutoffs_csv"], index=False, encoding="utf-8-sig")

    excel_tables = {
        "metadata": metadata_table,
        "daily_dataset_sample": pd.concat(
            [dataset.head(250), dataset.tail(250)]
        ).loc[lambda x: ~x.index.duplicated(keep="first")].reset_index(),
        "predictor_describe": predictor_describe,
        "predictor_pearson": predictor_pearson.reset_index(),
        "predictor_spearman": predictor_spearman.reset_index(),
        "quantile_cutoffs": cutoff_df,
        "quintile_results": quintile_results,
        "extreme_results": extreme_results,
        "all_results": all_results,
        "mean_vs_zero_results": mean_vs_zero_results,
        "all_results_v5": all_results_v5,
        "data_quality": data_quality,
        "target_definitions": target_definitions,
        "selected_stocks": selected_metadata,
        "security_filter_audit": full_metadata_audit,
        "figure_manifest": figure_manifest,
    }

    for validation_name, validation_df in validations.items():
        # Excel sheet name <=31 chars
        sheet_name = f"val_{validation_name}"[:31]
        excel_tables[sheet_name] = validation_df

    with pd.ExcelWriter(paths["summary_xlsx"], engine="openpyxl") as writer:
        for sheet_name, table in excel_tables.items():
            table_to_write = table.copy()
            if isinstance(table_to_write.index, pd.DatetimeIndex):
                table_to_write = table_to_write.reset_index()
            table_to_write.to_excel(writer, sheet_name=sheet_name, index=False)

        apply_excel_formats(writer, list(excel_tables.keys()))

    for label, path in paths.items():
        print(f"[SAVE] {label}: {path}")

    return paths

## 12. 執行完整分析

此 cell 是主流程。若 FinLab metadata 或還原價格 key 與候選名稱不同，程式會停在對應步驟並提供 columns/sample，請只修改 Config，不需改動其餘架構。

In [12]:
# ============================================================
# 11. Main pipeline
# ============================================================

def main(refresh: bool = REFRESH) -> Dict[str, Any]:
    context: Dict[str, Any] = {}

    steps = [
        "load_data",
        "filter_common_stocks",
        "build_market_breadth",
        "load_0050_prices",
        "build_forward_returns",
        "build_dataset",
        "data_quality",
        "build_groups",
        "run_statistics",
        "validations",
        "plots",
        "export",
        "final_summary",
    ]

    pbar = tqdm(steps, desc="Main pipeline")

    for step in pbar:
        pbar.set_postfix_str(step)
        print(f"\n\n========== STEP: {step} ==========")

        try:
            if step == "load_data":
                loaded = load_data(refresh=refresh)
                context.update(loaded)

            elif step == "filter_common_stocks":
                (
                    context["common_close"],
                    context["selected_metadata"],
                    context["full_metadata_audit"],
                ) = filter_common_stocks(
                    context["stock_close"],
                    context["security_metadata_raw"],
                )

            elif step == "build_market_breadth":
                # Cache 必須綁定實際股票母體，避免修改市場篩選後誤讀舊 breadth。
                universe_symbols = sorted(
                    context["common_close"].columns.astype(str).tolist()
                )
                universe_signature = hashlib.sha256(
                    "|".join(universe_symbols).encode("utf-8")
                ).hexdigest()[:12]
                breadth_cache = CACHE_DIR / (
                    f"market_breadth_common_stocks_v4_"
                    f"{len(universe_symbols)}_{universe_signature}.parquet"
                )

                rebuild_breadth = refresh or not breadth_cache.exists()

                if not rebuild_breadth:
                    print(f"[CACHE] market breadth: {breadth_cache}")
                    cached_breadth = pd.read_parquet(breadth_cache)
                    cached_breadth = normalize_datetime_index(
                        cached_breadth, "breadth_cache"
                    )

                    cached_total = (
                        int(cached_breadth["universe_total"].dropna().iloc[0])
                        if "universe_total" in cached_breadth.columns
                        and cached_breadth["universe_total"].notna().any()
                        else None
                    )

                    missing_cached_predictors = [
                        c for c in PREDICTORS if c not in cached_breadth.columns
                    ]

                    if missing_cached_predictors:
                        warnings.warn(
                            "市場廣度快取缺少目前版本所需 predictor，將自動重建："
                            f"{missing_cached_predictors}",
                            UserWarning,
                        )
                        rebuild_breadth = True
                    elif cached_total != len(universe_symbols):
                        warnings.warn(
                            "市場廣度快取的 universe_total 與目前篩選母體不一致，"
                            "將自動重建。"
                            f" cached={cached_total}, current={len(universe_symbols)}",
                            UserWarning,
                        )
                        rebuild_breadth = True
                    else:
                        context["breadth"] = cached_breadth

                if rebuild_breadth:
                    context["breadth"] = build_market_breadth(
                        context["common_close"],
                        reasonable_universe_total=len(universe_symbols),
                    )
                    context["breadth"].to_parquet(breadth_cache)
                    print(f"[CACHE SAVE] {breadth_cache}")

                print(
                    "[BREADTH UNIVERSE] "
                    f"stocks={len(universe_symbols)}, signature={universe_signature}"
                )

                context["predictor_quality"] = predictor_quality_table(
                    context["breadth"]
                )

            elif step == "load_0050_prices":
                (
                    context["price_0050"],
                    context["price_info"],
                ) = load_0050_prices(refresh=refresh)

            elif step == "build_forward_returns":
                context["price_returns"] = build_forward_returns(
                    context["price_0050"]
                )
                print(context["price_returns"].head(12))
                print(context["price_returns"].tail(12))

            elif step == "build_dataset":
                context["dataset"] = build_dataset(
                    context["breadth"],
                    context["price_returns"],
                )

            elif step == "data_quality":
                context["data_quality"] = build_data_quality_table(
                    context["dataset"]
                )
                context["predictor_describe"] = build_predictor_describe(
                    context["dataset"]
                )
                (
                    context["predictor_pearson"],
                    context["predictor_spearman"],
                ) = build_predictor_correlations(context["dataset"])

                print(context["data_quality"])
                print(context["predictor_describe"])
                print("\nPredictor Pearson correlation:")
                print(context["predictor_pearson"])
                print("\nPredictor Spearman correlation:")
                print(context["predictor_spearman"])

            elif step == "build_groups":
                (
                    context["groups"],
                    context["cutoff_df"],
                ) = build_groups(context["dataset"])
                print(context["cutoff_df"])

            elif step == "run_statistics":
                (
                    context["quintile_results"],
                    context["extreme_results"],
                    context["all_results"],
                ) = run_full_analysis(
                    context["dataset"],
                    context["groups"],
                )
                (
                    context["mean_vs_zero_results"],
                    context["mean_vs_zero_multiple_testing_summary"],
                ) = run_mean_vs_zero_analysis(
                    context["dataset"],
                    context["groups"],
                )
                context["all_results_v5"] = merge_all_results_v5(
                    context["all_results"],
                    context["mean_vs_zero_results"],
                )

            elif step == "validations":
                context["validations"] = run_validations(
                    common_close=context["common_close"],
                    breadth=context["breadth"],
                    price_returns=context["price_returns"],
                    dataset=context["dataset"],
                    groups=context["groups"],
                    all_results=context["all_results"],
                    mean_vs_zero_results=context["mean_vs_zero_results"],
                    all_results_v5=context["all_results_v5"],
                )

            elif step == "plots":
                context["figure_manifest"] = make_plots(
                    context["dataset"],
                    context["cutoff_df"],
                    context["quintile_results"],
                    context["mean_vs_zero_results"],
                )

            elif step == "export":
                context["metadata_table"] = build_metadata_table(
                    selected_keys=context["selected_keys"],
                    price_info=context["price_info"],
                    common_close=context["common_close"],
                    dataset=context["dataset"],
                )
                context["output_paths"] = export_results(
                    dataset=context["dataset"],
                    metadata_table=context["metadata_table"],
                    selected_metadata=context["selected_metadata"],
                    full_metadata_audit=context["full_metadata_audit"],
                    predictor_describe=context["predictor_describe"],
                    predictor_pearson=context["predictor_pearson"],
                    predictor_spearman=context["predictor_spearman"],
                    cutoff_df=context["cutoff_df"],
                    quintile_results=context["quintile_results"],
                    extreme_results=context["extreme_results"],
                    all_results=context["all_results"],
                    mean_vs_zero_results=context["mean_vs_zero_results"],
                    all_results_v5=context["all_results_v5"],
                    data_quality=context["data_quality"],
                    target_definitions=TARGET_METADATA,
                    validations=context["validations"],
                    figure_manifest=context["figure_manifest"],
                )

            elif step == "final_summary":
                dataset = context["dataset"]
                print("\n========== FINAL SUMMARY ==========")
                print(f"研究期間       : {dataset.index.min()} ~ {dataset.index.max()}")
                print(f"普通股數量     : {context['common_close'].shape[1]}")
                print(
                    "每日 valid_count: "
                    f"min={dataset['valid_count'].min()}, "
                    f"median={dataset['valid_count'].median():.0f}, "
                    f"max={dataset['valid_count'].max()}"
                )
                print(f"有效交易日數   : {dataset[PREDICTORS].notna().any(axis=1).sum()}")
                print("\n分位門檻:")
                print(
                    context["cutoff_df"].pivot(
                        index="predictor",
                        columns="cutoff_name",
                        values="cutoff_value",
                    )
                )
                print("\nMean vs Zero 多重比較摘要:")
                for name, summary in context["mean_vs_zero_multiple_testing_summary"].items():
                    print(f"  {name}: {summary}")
                print("\n輸出路徑:")
                for label, path in context["output_paths"].items():
                    print(f"  {label}: {path}")
                print("\n========== COMPLETED ==========")

        except Exception as exc:
            obj = context.get("dataset", context.get("stock_close"))
            raise_step_error(step, step, obj, exc)

    return context


RESULTS = main(refresh=REFRESH)

Main pipeline:   0%|                                                                 | 0/13 [00:00<?, ?it/s, load_data]



========== STEP: load_data ==========
[CACHE] stock_close_0_price_收盤價: C:\Users\hh483\Downloads\00 量化交易研究\cache\stock_close_0_price_收盤價.parquet
[LOADED] name=stock_close_0_price_收盤價, type=DataFrame, shape=(4726, 2761), index_range=2007-04-23 00:00:00 ~ 2026-07-15 00:00:00, columns_sample=['0015', '00400A', '00401A', '00402A', '00403A', '00404A', '00405A', '00406A', '00407A', '00408A', '0050', '0051']
[KEY SELECTED] stock_close -> price:收盤價

===== INSPECT: stock_close (price:收盤價) =====
name=stock_close (price:收盤價), type=DataFrame, shape=(3799, 2761), index_range=2011-01-03 00:00:00 ~ 2026-07-15 00:00:00, columns_sample=['0015', '00400A', '00401A', '00402A', '00403A', '00404A', '00405A', '00406A', '00407A', '00408A', '0050', '0051']
dtypes sample:
symbol
0015      float64
00400A    float64
00401A    float64
00402A    float64
00403A    float64
00404A    float64
00405A    float64
00406A    float64
00407A    float64
00408A    float64
0050      float64
0051      float64
dtype: object
head:

Main pipeline:   8%|███▌                                          | 1/13 [00:01<00:17,  1.48s/it, filter_common_stocks]C:\Users\hh483\AppData\Local\Temp\ipykernel_7904\4104315255.py:134: UserWarning: metadata 無法辨識 security_type 欄位；本次將依市場、名稱與代號規則篩選。請在輸出 metadata 與 sample 中人工核對。
  warnings.warn(




========== STEP: filter_common_stocks ==========

[METADATA MAPPING]
{'symbol': 'stock_id', 'name': '公司簡稱', 'market': '市場別', 'security_type': None}
  symbol security_name market
0   1101            台泥    sii
1   1102            亞泥    sii
2   1103            嘉泥    sii
3   1104            環泥    sii
4   1108            幸福    sii
5   1109            信大    sii
6   1110            東泥    sii
7   1201            味全    sii
8   1203            味王    sii
9   1210            大成    sii

[MARKET FILTER AUDIT]
        metadata_count  allowed_count  final_included_count
market                                                     
otc                891            891                   891
rotc               355              0                     0
sii               1089           1089                  1083


Main pipeline:  23%|███████████▌                                      | 3/13 [00:02<00:06,  1.62it/s, load_0050_prices]

...


========== VALIDATION SUMMARY ==========
manual_breadth: PASS
manual_forward_returns: PASS
group_proportions: PASS
group_partition: PASS
tail_nans: PASS
hac_beta_identity: PASS
no_inf: PASS
ratio_identities: PASS
timing_alignment: PASS
mean_vs_zero: PASS


========== STEP: plots ==========



Predictor histograms: 100%|██████████████████████████████████████████████████████████████| 6/6 [00:03<00:00,  1.75it/s]

Relation plots: 100%|██████████████████████████████████████████████████████████████████| 24/24 [00:12<00:00,  1.93it/s]

Quantile plots: 100%|██████████████████████████████████████████████████████████████████| 24/24 [00:07<00:00,  3.10it/s]

Excess-return plots: 100%|█████████████████████████████████████████████████████████████| 24/24 [00:10<00:00,  2.33it/s]

Main pipeline:  85%|█████████████████████████████████████████████████▉         | 11/13 [00:58<00:27, 13.69s/it, export]

[FIGURES] saved=102 to C:\Users\hh483\Downloads\00 量化交易研究\output\market_breadth_0050_study\figures


========== STEP: export ==========


Main pipeline: 100%|████████████████████████████████████████████████████| 13/13 [01:04<00:00,  4.98s/it, final_summary]

[SAVE] dataset_parquet: C:\Users\hh483\Downloads\00 量化交易研究\output\market_breadth_0050_study\market_breadth_dataset.parquet
[SAVE] dataset_csv: C:\Users\hh483\Downloads\00 量化交易研究\output\market_breadth_0050_study\market_breadth_dataset.csv
[SAVE] summary_xlsx: C:\Users\hh483\Downloads\00 量化交易研究\output\market_breadth_0050_study\market_breadth_summary.xlsx
[SAVE] cutoffs_csv: C:\Users\hh483\Downloads\00 量化交易研究\output\market_breadth_0050_study\quantile_cutoffs.csv
[SAVE] figures_dir: C:\Users\hh483\Downloads\00 量化交易研究\output\market_breadth_0050_study\figures


========== STEP: final_summary ==========

========== FINAL SUMMARY ==========
研究期間       : 2011-01-03 00:00:00 ~ 2026-07-15 00:00:00
普通股數量     : 1974
每日 valid_count: min=0, median=1543, max=1955
有效交易日數   : 3799

分位門檻:
cutoff_name                  p05         p20         p40         p60         p80          p95
predictor                                                                                    
breadth_log_ad_ratio   -1.69644

## 13. 最終判讀提醒

> **本分析為 exploratory analysis。分位門檻使用完整樣本期間估計，含有樣本內資訊，只用於探索市場廣度與未來報酬的關係形狀，不視為可直接交易的樣本外績效。**
>
> **對 +2、+3、+5、+10 日重疊報酬，以 HAC t-value 與 HAC p-value 為主要統計推論；普通 t-test、普通 p-value 與 annualized Sharpe 僅作描述。**

### 研究結論規則

本 notebook 本身不因單次樣本內顯著而宣告策略有效。下一階段至少需要：

- expanding / rolling quantile 門檻；
- train / validation / test 或 walk-forward；
- 不同市場狀態與年度穩定性；
- 交易成本、滑價與可執行時點；
- multiple testing / data snooping 控制；
- 檢查少數年份或極端事件是否主導結果。

在完成樣本外驗證前，結論只能是：**修改後再測**。

## v5 判讀提醒

本 notebook 同時回答：

1. Group vs Non-group：訊號日是否相對其他日期有較高或較低報酬。
2. Group Mean vs Zero：訊號日之後的平均報酬本身是否為正或負。

即使 group mean 為正，也不代表優於一般市場日；即使 group vs non-group 顯著，也不代表報酬本身為正。多日重疊報酬以 HAC 為主要推論；binomial test 未處理時間依賴，只作為勝率補充。所有檢定仍屬完整樣本門檻的 exploratory analysis，尚未包含交易成本、滑價、walk-forward 或 out-of-sample validation。
